In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - textanalyzetext
====================================

text: 1. textgenetextcalculatecoverage
2. textfold changetext
3. text0/1text

text: - microbegenetext, textwithtext
- textgene(targettextnormal)
- textandgene cluster
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import time

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    # path
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    # Targetsamplepath
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    TARGET_BAM_TXT_DIR: str = "results_V3/special/CRC/03.coverage/R"
    # Normalsamplepath
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    # outputpath
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50 # minimum gene length(bp)
    MIN_TARGET_COVERAGE: float = 1.0 # minimum target average coverage
    MIN_FOLD_CHANGE: float = 2.0 # textfold change (target/normal)
    MAX_NORMAL_COVERAGE: float = 0.5 # maximum normal average coverage(optional)
    # coveragecalculatetext
    MIN_GENE_COVERAGE_FRACTION: float = 0.3 # at least 30% of the gene region has coverage
    def __post_init__(self):
        """buildtextpath"""
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'NORMAL_COVERAGE_DIR', 'OUTPUT_DIR']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== Utility functions ====================

def load_bedgraph(bedgraph_file: str) -> Dict[str, List[Tuple[int, int, float]]]:
    """
    loadbedgraphfile
    return: {chromosome: [(start, end, coverage), ...]}
    """
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        print(f" warning: bedgraphfiletextexists: {bedgraph_file}")
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                coverage = float(parts[3])
                regions[chrom].append((start, end, coverage))
                return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
"""
calculategenetextcoveragestatistics
return:
{
'mean_coverage': averagecoverage,
'covered_bases': withcoveragetext,
'total_bases': genetext,
'coverage_fraction': coveragetext
}
"""
gene_length = gene_end - gene_start
# texteachtextcoverage
coverage_array = np.zeros(gene_length, dtype=float)
# textcoverage
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    # calculatetext
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        # textfortextgenetext
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        # calculatestatisticstext
        covered_bases = np.sum(coverage_array > 0)
        mean_coverage = np.mean(coverage_array)
        coverage_fraction = covered_bases / gene_length if gene_length > 0 else 0
        return {
        'mean_coverage': mean_coverage,
        'covered_bases': int(covered_bases),
        'total_bases': gene_length,
        'coverage_fraction': coverage_fraction
    }

# ==================== annotationfiletext ====================

def parse_gff_line(line: str) -> Optional[Dict]:
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'source': fields[1],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string: str) -> Dict[str, str]:
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string: str) -> Dict[str, str]:
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotations(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """
    loadannotationfile
    return: {chromosome: [gene_info, ...]}
    """
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    parse_attrs = parse_gtf_attributes if file_type == 'gtf' else parse_gff_attributes
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if not record:
                continue
            # onlytextgenetext(orCDStext)
            if record['type'] not in ['gene', 'CDS', 'mRNA', 'transcript']:
                continue
            attrs = parse_attrs(record['attributes'])
            gene_info = {
                'seqid': record['seqid'],
                'type': record['type'],
                'start': record['start'],
                'end': record['end'],
                'strand': record['strand'],
                'length': record['end'] - record['start'],
            }
            # extractgeneIDandtext
            if file_type == 'gtf':
                gene_info['gene_id'] = attrs.get('gene_id', 'N/A')
                gene_info['gene_name'] = attrs.get('gene_name', attrs.get('gene_id', 'N/A'))
                gene_info['gene_biotype'] = attrs.get('gene_biotype', 'N/A')
                gene_info['product'] = attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            else: # gff
                gene_info['gene_id'] = attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A')))
                gene_info['gene_name'] = attrs.get('Name', attrs.get('gene', 'N/A'))
                gene_info['product'] = attrs.get('product', 'N/A')
                gene_info['locus_tag'] = attrs.get('locus_tag', 'N/A')
                annotations[record['seqid']].append(gene_info)
                return annotations

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    """
    deduplicategene(textgenetextfeature, text)
    """
    gene_dict = {}
    for gene in genes:
        # filtertextgene
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict:
            gene_dict[gene_id] = gene
        else:
            # textfeature
            if gene['length'] > gene_dict[gene_id]['length']:
                gene_dict[gene_id] = gene
                return list(gene_dict.values())

# ==================== textprocesstext ====================

def process_taxon(taxon: str, sample_id: str = None) -> pd.DataFrame:
    """
    processtextmicrobesallsample(ortextsample)
    text:
    taxon: microbetext
    sample_id: textsampleID(optional, text)
    return:
    DataFrametextgeneinformation
    """
    print(f"\n{'='*70}")
    print(f"processmicrobe: {taxon}")
    print(f"{'='*70}")
    # 1. loadannotationfile
    print("\n[1/5] loadgeneannotation...")
    gff_file = os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz")
    gtf_file = os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz")
    import glob
    gff_files = glob.glob(gff_file)
    gtf_files = glob.glob(gtf_file)
    annotations = {}
    file_type = None
    if gff_files:
        print(f" textGFF: {os.path.basename(gff_files[0])}")
        annotations = load_annotations(gff_files[0], 'gff')
        file_type = 'gff'
    elif gtf_files:
        print(f" textGTF: {os.path.basename(gtf_files[0])}")
        annotations = load_annotations(gtf_files[0], 'gtf')
        file_type = 'gtf'
    else:
        print(f" error: not foundannotationfile")
        return pd.DataFrame()
    # deduplicategene
    all_genes = []
    for chrom, genes in annotations.items():
        all_genes.extend(genes)
        unique_genes = deduplicate_genes(all_genes)
        print(f" loadtext {len(unique_genes)} genes")
        # 2. loadNormalsamplecoverage
        print("\n[2/5] loadNormalcoverage...")
        normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'coverage_Normal.bedgraph')
        normal_coverage = load_bedgraph(normal_bedgraph)
        print(f" Normalcoveragetext: {sum(len(v) for v in normal_coverage.values())}")
        # 3. findTargetsample
        print("\n[3/5] findTargetsample...")
        target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
        if not os.path.exists(target_dir):
            print(f" error: Targetdirectorytextexists: {target_dir}")
            return pd.DataFrame()
        bedgraph_files = [f for f in os.listdir(target_dir)             if f.endswith('_rna_output.pathseq.bedgraph')]
        if sample_id:
            # onlyprocesstextsample
            bedgraph_files = [f for f in bedgraph_files if f.startswith(f"{sample_id}_")]
            print(f" textsample: {sample_id}")
            print(f" found {len(bedgraph_files)} Targetsample")
            # 4. processtextsamples
            print("\n[4/5] calculategenecoverageandtext...")
            all_results = []
            for idx, bedgraph_file in enumerate(bedgraph_files, 1):
                sample_name = bedgraph_file.replace('_rna_output.pathseq.bedgraph', '')
                print(f"\n [{idx}/{len(bedgraph_files)}] sample: {sample_name}")
                # loadTargetcoverage
                target_bedgraph = os.path.join(target_dir, bedgraph_file)
                target_coverage = load_bedgraph(target_bedgraph)
                sample_results = []
                for gene in unique_genes:
                    seqid = gene['seqid']
                    # calculateTargetcoverage
                    target_stats = calculate_gene_coverage(
                        gene['start'], gene['end'],
                        target_coverage.get(seqid, [])
)
# calculateNormalcoverage
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(seqid, [])
)
# filtertext
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue # insufficient target coverage
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue # target coverage too low
    # calculatefold change (text)
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue # insufficient fold change
    # optional: filterNormalcoveragetextgene
    if config.MAX_NORMAL_COVERAGE and normal_stats['mean_coverage'] > config.MAX_NORMAL_COVERAGE:
        continue
    # saveresults
    result = {
        'sample': sample_name,
        'taxon': taxon,
        'seqid': seqid,
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        # Targetcoveragestatistics
        'target_mean_cov': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        # Normalcoveragestatistics
        'normal_mean_cov': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        # text
        'fold_change': round(fold_change, 2),
        'coverage_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    }
    if file_type == 'gtf':
        result['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        result['locus_tag'] = gene.get('locus_tag', 'N/A')
        sample_results.append(result)
        print(f" found {len(sample_results)} textgene")
        all_results.extend(sample_results)
        # 5. textresults
        print(f"\n[5/5] text: {len(all_results)} textgene")
        if not all_results:
            return pd.DataFrame()
        df = pd.DataFrame(all_results)
        df = df.sort_values(['sample', 'fold_change'], ascending=[True, False])
        return df

def extract_gene_sequences(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """
    fromtextgenetextextractgenesequence
    return: {gene_key: sequence}
    """
    print(f"\nextractgenesequence...")
    # loadtextgenetext
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        print(f" warning: textgenetextexists: {ref_file}")
        return {}
    genome_seqs = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        print(f" loadtext {len(genome_seqs)} textsequence")
        # extractgenesequence
        gene_sequences = {}
        for _, row in df.iterrows():
            seqid = row['seqid']
            start = row['gene_start']
            end = row['gene_end']
            if seqid not in genome_seqs:
                continue
            sequence = genome_seqs[seqid][start-1:end]
            gene_key = f"{row['sample']}|{seqid}_{start}_{end}"
            gene_sequences[gene_key] = sequence
            print(f" extracttext {len(gene_sequences)} genessequence")
            return gene_sequences

def save_results(df: pd.DataFrame, taxon: str, sample_id: str = None):
    """saveresults"""
    if df.empty:
        print("\ntextwithresultstextsave")
        return
    # Create output directory
    output_dir = os.path.join(config.OUTPUT_DIR, 'reports')
    os.makedirs(output_dir, exist_ok=True)
    # saveCSV
    if sample_id:
        csv_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.csv")
    else:
        csv_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.csv")
        df.to_csv(csv_file, index=False)
        print(f"\nOK resultssaved: {csv_file}")
        # extracttextsavesequence
        gene_sequences = extract_gene_sequences(df, taxon)
        if gene_sequences:
            if sample_id:
                fna_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.fna")
            else:
                fna_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.fna")
                with open(fna_file, 'w') as f:
                    for gene_key, sequence in gene_sequences.items():
                        parts = gene_key.split('|')
                        sample = parts[0]
                        coords = parts[1]
                        # fromdfintextgeneinformation
                        mask = (df['sample'] == sample) & (df['seqid'] + '_' + df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str) == coords)
                        if mask.any():
                            gene_info = df[mask].iloc[0]
                            header = f">{coords} {gene_info['gene_name']} | {gene_info['product']} | FC={gene_info['fold_change']}"
                        else:
                            header = f">{coords}"
                            f.write(header + '\n')
                            # textrows60text
                            for i in range(0, len(sequence), 60):
                                f.write(sequence[i:i+60] + '\n')
                                print(f"OK sequencesaved: {fna_file}")
                                # textstatisticstext
                                print(f"\n{'='*70}")
                                print("resultstext")
                                print(f"{'='*70}")
                                print(f"microbe: {taxon}")
                                print(f"sample count: {df['sample'].nunique()}")
                                print(f"textgenetext: {len(df)}")
                                print(f"\nFold Changetext:")
                                print(df['fold_change'].describe())
                                print(f"\ntextsamplestatistics:")
                                print(df.groupby('sample').size().sort_values(ascending=False))

# ==================== text ====================

def main():
    """text"""
    print("=" * 70)
    print("MTRgenefilter - textanalyzetext")
    print("=" * 70)
    print("\ntext:")
    print(f" minimum gene length: {config.MIN_GENE_LENGTH} bp")
    print(f" textTargetcoverage: {config.MIN_TARGET_COVERAGE}")
    print(f" textFold Change: {config.MIN_FOLD_CHANGE}")
    print(f" genecoveragetext: {config.MIN_GENE_COVERAGE_FRACTION}")
    if config.MAX_NORMAL_COVERAGE:
        print(f" textNormalcoverage: {config.MAX_NORMAL_COVERAGE}")
        # text: onlyprocessB7sample
        print("\n" + "="*70)
        print("text: onlyprocessB7sample")
        print("="*70)
        # readmicrobe list
        taxon_df = pd.read_csv(os.path.join(config.BASE_DIR, config.TAXON_LIST))
        taxons = taxon_df['Taxon'].tolist()
        print(f"\nshared {len(taxons)} microbes, text...")
        # textmicrobesB7sample
        test_taxon = taxons[0]
        start_time = time.time()
        df = process_taxon(test_taxon, sample_id='B7')
        if not df.empty:
            save_results(df, test_taxon, sample_id='B7')
            elapsed = time.time() - start_time
            print(f"\ntext: {elapsed:.2f} text")
            print("\n" + "="*70)
            print("textcompleted!textcheckresults, textreasonableaftertextprocesstextdata")
            print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - textanalyzetext
====================================

text: 1. textgenetextcalculatecoverage
2. **normalizetotext** (RPM: Reads Per Million)
3. textfold changetext
4. text0/1text

text: - microbegenetext, textwithtext
- textgene(targettextnormal)
- textandgene cluster

normalizetext: - Target: textsamplesreads
- Normal: 89samplestextreads
- text: textnormalizeto1M reads (RPM)
- textcomparetext"text"text"textreadstext"
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import time

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    # path
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    # Targetsamplepath
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    TARGET_BAM_TXT_DIR: str = "results_V3/special/CRC/03.coverage/R"
    # Normalsamplepath
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    # outputpath
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50 # minimum gene length(bp)
    MIN_TARGET_COVERAGE: float = 0.5 # minimum target average coverage (RPM)
    MIN_FOLD_CHANGE: float = 1.5 # textfold change (target/normal) - can be more permissive after normalization
    MAX_NORMAL_COVERAGE: float = None # maximum normal average coverage(None=no limit)
    # coveragecalculatetext
    MIN_GENE_COVERAGE_FRACTION: float = 0.3 # at least 30% of the gene region has coverage
    def __post_init__(self):
        """buildtextpath"""
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'NORMAL_COVERAGE_DIR', 'OUTPUT_DIR']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== Utility functions ====================

def get_total_mapped_reads(bam_txt_file: str) -> int:
    """
    from.bam.txtfilestatisticstextmapped readstext
    """
    if not os.path.exists(bam_txt_file):
        return 0
    read_count = 0
    with open(bam_txt_file, 'r') as f:
        for line in f:
            if not line.startswith('@'):
                read_count += 1
                return read_count

def load_bedgraph(bedgraph_file: str, normalize_factor: float = 1.0) -> Dict[str, List[Tuple[int, int, float]]]:
    """
    loadbedgraphfile, textrowsnormalize
    text:
    bedgraph_file: bedgraphfilepath
    normalize_factor: normalization factor (targettext / actualtext)
    return: {chromosome: [(start, end, normalized_coverage), ...]}
    """
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        print(f" warning: bedgraphfiletextexists: {bedgraph_file}")
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                coverage = float(parts[3]) * normalize_factor # normalize
                regions[chrom].append((start, end, coverage))
                return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
"""
calculategenetextcoveragestatistics
return:
{
'mean_coverage': averagecoverage,
'covered_bases': withcoveragetext,
'total_bases': genetext,
'coverage_fraction': coveragetext
}
"""
gene_length = gene_end - gene_start
# texteachtextcoverage
coverage_array = np.zeros(gene_length, dtype=float)
# textcoverage
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    # calculatetext
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        # textfortextgenetext
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        # calculatestatisticstext
        covered_bases = np.sum(coverage_array > 0)
        mean_coverage = np.mean(coverage_array)
        coverage_fraction = covered_bases / gene_length if gene_length > 0 else 0
        return {
        'mean_coverage': mean_coverage,
        'covered_bases': int(covered_bases),
        'total_bases': gene_length,
        'coverage_fraction': coverage_fraction
    }

# ==================== annotationfiletext ====================

def parse_gff_line(line: str) -> Optional[Dict]:
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'source': fields[1],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string: str) -> Dict[str, str]:
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string: str) -> Dict[str, str]:
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotations(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """
    loadannotationfile
    return: {chromosome: [gene_info, ...]}
    """
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    parse_attrs = parse_gtf_attributes if file_type == 'gtf' else parse_gff_attributes
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if not record:
                continue
            # onlytextgenetext(orCDStext)
            if record['type'] not in ['gene', 'CDS', 'mRNA', 'transcript']:
                continue
            attrs = parse_attrs(record['attributes'])
            gene_info = {
                'seqid': record['seqid'],
                'type': record['type'],
                'start': record['start'],
                'end': record['end'],
                'strand': record['strand'],
                'length': record['end'] - record['start'],
            }
            # extractgeneIDandtext
            if file_type == 'gtf':
                gene_info['gene_id'] = attrs.get('gene_id', 'N/A')
                gene_info['gene_name'] = attrs.get('gene_name', attrs.get('gene_id', 'N/A'))
                gene_info['gene_biotype'] = attrs.get('gene_biotype', 'N/A')
                gene_info['product'] = attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            else: # gff
                gene_info['gene_id'] = attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A')))
                gene_info['gene_name'] = attrs.get('Name', attrs.get('gene', 'N/A'))
                gene_info['product'] = attrs.get('product', 'N/A')
                gene_info['locus_tag'] = attrs.get('locus_tag', 'N/A')
                annotations[record['seqid']].append(gene_info)
                return annotations

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    """
    deduplicategene(textgenetextfeature, text)
    """
    gene_dict = {}
    for gene in genes:
        # filtertextgene
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict:
            gene_dict[gene_id] = gene
        else:
            # textfeature
            if gene['length'] > gene_dict[gene_id]['length']:
                gene_dict[gene_id] = gene
                return list(gene_dict.values())

# ==================== textprocesstext ====================

def process_taxon(taxon: str, sample_id: str = None) -> pd.DataFrame:
    """
    processtextmicrobesallsample(ortextsample)
    text:
    taxon: microbetext
    sample_id: textsampleID(optional, text)
    return:
    DataFrametextgeneinformation
    """
    print(f"\n{'='*70}")
    print(f"processmicrobe: {taxon}")
    print(f"{'='*70}")
    # 1. loadannotationfile
    print("\n[1/6] loadgeneannotation...")
    gff_file = os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz")
    gtf_file = os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz")
    import glob
    gff_files = glob.glob(gff_file)
    gtf_files = glob.glob(gtf_file)
    annotations = {}
    file_type = None
    if gff_files:
        print(f" textGFF: {os.path.basename(gff_files[0])}")
        annotations = load_annotations(gff_files[0], 'gff')
        file_type = 'gff'
    elif gtf_files:
        print(f" textGTF: {os.path.basename(gtf_files[0])}")
        annotations = load_annotations(gtf_files[0], 'gtf')
        file_type = 'gtf'
    else:
        print(f" error: not foundannotationfile")
        return pd.DataFrame()
    # deduplicategene
    all_genes = []
    for chrom, genes in annotations.items():
        all_genes.extend(genes)
        unique_genes = deduplicate_genes(all_genes)
        print(f" loadtext {len(unique_genes)} genes")
        # 2. calculateNormalsampletext
        print("\n[2/6] calculatetext...")
        normal_bam_txt = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'merged_Normal_sorted.bam.txt')
        normal_total_reads = get_total_mapped_reads(normal_bam_txt)
        if normal_total_reads == 0:
            print(f" warning: notextNormal readstext, textnormalizedata")
            normal_norm_factor = 1.0
        else:
            print(f" Normaltextreads: {normal_total_reads:,}")
            # normalizeto1M reads
            normal_norm_factor = 1e6 / normal_total_reads
            print(f" Normalnormalization factor: {normal_norm_factor:.6f}")
            # 3. loadNormalsamplecoverage(after normalization)
            print("\n[3/6] loadNormalcoverage...")
            normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,                 'coverage_Normal.bedgraph')
            normal_coverage = load_bedgraph(normal_bedgraph, normal_norm_factor)
            print(f" Normalcoveragetext: {sum(len(v) for v in normal_coverage.values())}")
            # 4. findTargetsample
            print("\n[4/6] findTargetsample...")
            target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
            if not os.path.exists(target_dir):
                print(f" error: Targetdirectorytextexists: {target_dir}")
                return pd.DataFrame()
            bedgraph_files = [f for f in os.listdir(target_dir)                 if f.endswith('_rna_output.pathseq.bedgraph')]
            if sample_id:
                # onlyprocesstextsample
                bedgraph_files = [f for f in bedgraph_files if f.startswith(f"{sample_id}_")]
                print(f" textsample: {sample_id}")
                print(f" found {len(bedgraph_files)} Targetsample")
                # 5. processtextsamples
                print("\n[5/6] calculategenecoverageandtext(after normalization)...")
                all_results = []
                for idx, bedgraph_file in enumerate(bedgraph_files, 1):
                    sample_name = bedgraph_file.replace('_rna_output.pathseq.bedgraph', '')
                    print(f"\n [{idx}/{len(bedgraph_files)}] sample: {sample_name}")
                    # calculateTargetsampletext
                    target_bam_txt = os.path.join(target_dir,                         f"{sample_name}_rna_output.pathseq_sorted.bam.txt")
                    target_total_reads = get_total_mapped_reads(target_bam_txt)
                    if target_total_reads == 0:
                        print(f" warning: notextTarget readstext, skiptextsample")
                        continue
                    print(f" Targettextreads: {target_total_reads:,}")
                    target_norm_factor = 1e6 / target_total_reads
                    print(f" Targetnormalization factor: {target_norm_factor:.6f}")
                    # loadTargetcoverage(after normalization)
                    target_bedgraph = os.path.join(target_dir, bedgraph_file)
                    target_coverage = load_bedgraph(target_bedgraph, target_norm_factor)
                    sample_results = []
                    for gene in unique_genes:
                        seqid = gene['seqid']
                        # calculateTargetcoverage
                        target_stats = calculate_gene_coverage(
                            gene['start'], gene['end'],
                            target_coverage.get(seqid, [])
)
# calculateNormalcoverage
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(seqid, [])
)
# filtertext
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue # insufficient target coverage
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue # target coverage too low
    # calculatefold change (text)
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue # insufficient fold change
    # optional: filterNormalcoveragetextgene
    if config.MAX_NORMAL_COVERAGE and normal_stats['mean_coverage'] > config.MAX_NORMAL_COVERAGE:
        continue
    # saveresults
    result = {
        'sample': sample_name,
        'taxon': taxon,
        'seqid': seqid,
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        # Targetcoveragestatistics (after normalizationRPM)
        'target_rpm': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        # Normalcoveragestatistics (after normalizationRPM)
        'normal_rpm': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        # text
        'fold_change': round(fold_change, 2),
        'rpm_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    }
    if file_type == 'gtf':
        result['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        result['locus_tag'] = gene.get('locus_tag', 'N/A')
        sample_results.append(result)
        print(f" found {len(sample_results)} textgene")
        all_results.extend(sample_results)
        # 6. textresults
        print(f"\n[6/6] text: {len(all_results)} textgene")
        if not all_results:
            return pd.DataFrame()
        df = pd.DataFrame(all_results)
        df = df.sort_values(['sample', 'fold_change'], ascending=[True, False])
        return df

def extract_gene_sequences(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """
    fromtextgenetextextractgenesequence
    return: {gene_key: sequence}
    """
    print(f"\nextractgenesequence...")
    # loadtextgenetext
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        print(f" warning: textgenetextexists: {ref_file}")
        return {}
    genome_seqs = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        print(f" loadtext {len(genome_seqs)} textsequence")
        # extractgenesequence
        gene_sequences = {}
        for _, row in df.iterrows():
            seqid = row['seqid']
            start = row['gene_start']
            end = row['gene_end']
            if seqid not in genome_seqs:
                continue
            sequence = genome_seqs[seqid][start-1:end]
            gene_key = f"{row['sample']}|{seqid}_{start}_{end}"
            gene_sequences[gene_key] = sequence
            print(f" extracttext {len(gene_sequences)} genessequence")
            return gene_sequences

def save_results(df: pd.DataFrame, taxon: str, sample_id: str = None):
    """saveresults"""
    if df.empty:
        print("\ntextwithresultstextsave")
        return
    # Create output directory
    output_dir = os.path.join(config.OUTPUT_DIR, 'reports')
    os.makedirs(output_dir, exist_ok=True)
    # saveCSV
    if sample_id:
        csv_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.csv")
    else:
        csv_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.csv")
        df.to_csv(csv_file, index=False)
        print(f"\nOK resultssaved: {csv_file}")
        # extracttextsavesequence
        gene_sequences = extract_gene_sequences(df, taxon)
        if gene_sequences:
            if sample_id:
                fna_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.fna")
            else:
                fna_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.fna")
                with open(fna_file, 'w') as f:
                    for gene_key, sequence in gene_sequences.items():
                        parts = gene_key.split('|')
                        sample = parts[0]
                        coords = parts[1]
                        # fromdfintextgeneinformation
                        mask = (df['sample'] == sample) & (df['seqid'] + '_' + df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str) == coords)
                        if mask.any():
                            gene_info = df[mask].iloc[0]
                            header = f">{coords} {gene_info['gene_name']} | {gene_info['product']} | FC={gene_info['fold_change']}"
                        else:
                            header = f">{coords}"
                            f.write(header + '\n')
                            # textrows60text
                            for i in range(0, len(sequence), 60):
                                f.write(sequence[i:i+60] + '\n')
                                print(f"OK sequencesaved: {fna_file}")
                                # textstatisticstext
                                print(f"\n{'='*70}")
                                print("resultstext")
                                print(f"{'='*70}")
                                print(f"microbe: {taxon}")
                                print(f"sample count: {df['sample'].nunique()}")
                                print(f"textgenetext: {len(df)}")
                                print(f"\nFold Changetext:")
                                print(df['fold_change'].describe())
                                print(f"\nTarget RPMtext:")
                                print(df['target_rpm'].describe())
                                print(f"\nNormal RPMtext:")
                                print(df['normal_rpm'].describe())
                                print(f"\ntextsamplestatistics:")
                                print(df.groupby('sample').size().sort_values(ascending=False))

# ==================== text ====================

def main():
    """text"""
    print("=" * 70)
    print("MTRgenefilter - textanalyzetext (normalize)")
    print("=" * 70)
    print("\ntext:")
    print(f" minimum gene length: {config.MIN_GENE_LENGTH} bp")
    print(f" textTargetcoverage: {config.MIN_TARGET_COVERAGE} RPM")
    print(f" textFold Change: {config.MIN_FOLD_CHANGE}")
    print(f" genecoveragetext: {config.MIN_GENE_COVERAGE_FRACTION}")
    if config.MAX_NORMAL_COVERAGE:
        print(f" textNormalcoverage: {config.MAX_NORMAL_COVERAGE} RPM")
        print("\nnormalizetext:")
        print(" - allsamplenormalizeto1M reads (RPM)")
        print(" - texttarget (textsample) vs normal (89sampletext) text")
        print(" - comparetext, textreadstext")
        # text: onlyprocessB7sample
        print("\n" + "="*70)
        print("text: onlyprocessB7sample")
        print("="*70)
        # readmicrobe list
        taxon_df = pd.read_csv(os.path.join(config.BASE_DIR, config.TAXON_LIST))
        taxons = taxon_df['Taxon'].tolist()
        print(f"\nshared {len(taxons)} microbes, text...")
        # textmicrobesB7sample
        test_taxon = taxons[0]
        start_time = time.time()
        df = process_taxon(test_taxon, sample_id='B7')
        if not df.empty:
            save_results(df, test_taxon, sample_id='B7')
            elapsed = time.time() - start_time
            print(f"\ntext: {elapsed:.2f} text")
            print("\n" + "="*70)
            print("textcompleted!textcheckresults, textreasonableaftertextprocesstextdata")
            print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - textanalyzetext
====================================

text: 1. textgenetextcalculatecoverage
2. **normalizetotext** (textRPM, textcoveragedata)
3. textfold changetext
4. text0/1text

normalizetext: - fromflagstatfilereadtext(paired in sequencing)
- Target: textsamplestext
- Normal: 89samplestextand
- normalization factor = 1,000,000 / text
- normalizecoverage = textcoverage × normalization factor

fortextcoveragenormalizetextRPKM/FPKM?
- bedgraphtextcoverage(reads depth)
- calculatetextaveragecoverage, textgenetext
- onlytextnormalizetextcompare
- text"normalized coverage per million reads"

text: - microbegenetext, textwithtext
- textgene(targettextnormal)
- textandgene cluster
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import time

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    # path
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    # Targetsamplepath
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    TARGET_FLAGSTAT_DIR: str = "data/special/CRC/R/flagstat"
    # Normalsamplepath
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    NORMAL_FLAGSTAT_DIR: str = "data/cancers_V1/CRC/Normal/flagstat"
    # outputpath
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50 # minimum gene length(bp)
    MIN_TARGET_COVERAGE: float = 0.5 # minimum target average coverage(after normalization)
    MIN_FOLD_CHANGE: float = 1.5 # textfold change (target/normal)
    MAX_NORMAL_COVERAGE: float = None # maximum normal average coverage(None=no limit)
    # coveragecalculatetext
    MIN_GENE_COVERAGE_FRACTION: float = 0.3 # at least 30% of the gene region has coverage
    def __post_init__(self):
        """buildtextpath"""
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'TARGET_FLAGSTAT_DIR',
            'NORMAL_COVERAGE_DIR', 'NORMAL_FLAGSTAT_DIR', 'OUTPUT_DIR']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== Utility functions ====================

def get_library_size_from_flagstat(flagstat_file: str) -> int:
    """
    fromflagstatfilereadtext(text6rowspaired in sequencingtext)
    textrows: 100732854 + 0 paired in sequencing
    return: 100732854
    """
    if not os.path.exists(flagstat_file):
        print(f" warning: flagstatfiletextexists: {flagstat_file}")
        return 0
    try:
        with open(flagstat_file, 'r') as f:
            lines = f.readlines()
            if len(lines) >= 6:
                # text6rows(text5)
                line = lines[5].strip()
                # extracttext
                value = int(line.split()[0])
                return value
    except Exception as e:
        print(f" error: readflagstatfailed: {e}")
        return 0
    return 0

def get_normal_total_library_size() -> int:
    """
    calculateNormalsampletext(89samplestextand)
    """
    import glob
    flagstat_files = glob.glob(os.path.join(config.NORMAL_FLAGSTAT_DIR,         "*_rna_hg38_align_sort.bam.flagstat.txt"))
    if not flagstat_files:
        print(f" warning: not foundNormal flagstatfile")
        return 0
    total_size = 0
    valid_samples = 0
    for flagstat_file in flagstat_files:
        size = get_library_size_from_flagstat(flagstat_file)
        if size > 0:
            total_size += size
            valid_samples += 1
            print(f" Normalsample count: {valid_samples}")
            print(f" Normaltext: {total_size:,}")
            return total_size

def get_total_mapped_reads(bam_txt_file: str) -> int:
    """
    from.bam.txtfilestatisticstextmapped readstext(text, textflagstat)
    """
    if not os.path.exists(bam_txt_file):
        return 0
    read_count = 0
    with open(bam_txt_file, 'r') as f:
        for line in f:
            if not line.startswith('@'):
                read_count += 1
                return read_count

def load_bedgraph(bedgraph_file: str, normalize_factor: float = 1.0) -> Dict[str, List[Tuple[int, int, float]]]:
    """
    loadbedgraphfile, textrowsnormalize
    text:
    bedgraph_file: bedgraphfilepath
    normalize_factor: normalization factor (targettext / actualtext)
    return: {chromosome: [(start, end, normalized_coverage), ...]}
    """
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        print(f" warning: bedgraphfiletextexists: {bedgraph_file}")
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                coverage = float(parts[3]) * normalize_factor # normalize
                regions[chrom].append((start, end, coverage))
                return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
"""
calculategenetextcoveragestatistics
return:
{
'mean_coverage': averagecoverage,
'covered_bases': withcoveragetext,
'total_bases': genetext,
'coverage_fraction': coveragetext
}
"""
gene_length = gene_end - gene_start
# texteachtextcoverage
coverage_array = np.zeros(gene_length, dtype=float)
# textcoverage
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    # calculatetext
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        # textfortextgenetext
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        # calculatestatisticstext
        covered_bases = np.sum(coverage_array > 0)
        mean_coverage = np.mean(coverage_array)
        coverage_fraction = covered_bases / gene_length if gene_length > 0 else 0
        return {
        'mean_coverage': mean_coverage,
        'covered_bases': int(covered_bases),
        'total_bases': gene_length,
        'coverage_fraction': coverage_fraction
    }

# ==================== annotationfiletext ====================

def parse_gff_line(line: str) -> Optional[Dict]:
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'source': fields[1],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string: str) -> Dict[str, str]:
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string: str) -> Dict[str, str]:
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotations(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """
    loadannotationfile
    return: {chromosome: [gene_info, ...]}
    """
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    parse_attrs = parse_gtf_attributes if file_type == 'gtf' else parse_gff_attributes
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if not record:
                continue
            # onlytextgenetext(orCDStext)
            if record['type'] not in ['gene', 'CDS', 'mRNA', 'transcript']:
                continue
            attrs = parse_attrs(record['attributes'])
            gene_info = {
                'seqid': record['seqid'],
                'type': record['type'],
                'start': record['start'],
                'end': record['end'],
                'strand': record['strand'],
                'length': record['end'] - record['start'],
            }
            # extractgeneIDandtext
            if file_type == 'gtf':
                gene_info['gene_id'] = attrs.get('gene_id', 'N/A')
                gene_info['gene_name'] = attrs.get('gene_name', attrs.get('gene_id', 'N/A'))
                gene_info['gene_biotype'] = attrs.get('gene_biotype', 'N/A')
                gene_info['product'] = attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            else: # gff
                gene_info['gene_id'] = attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A')))
                gene_info['gene_name'] = attrs.get('Name', attrs.get('gene', 'N/A'))
                gene_info['product'] = attrs.get('product', 'N/A')
                gene_info['locus_tag'] = attrs.get('locus_tag', 'N/A')
                annotations[record['seqid']].append(gene_info)
                return annotations

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    """
    deduplicategene(textgenetextfeature, text)
    """
    gene_dict = {}
    for gene in genes:
        # filtertextgene
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict:
            gene_dict[gene_id] = gene
        else:
            # textfeature
            if gene['length'] > gene_dict[gene_id]['length']:
                gene_dict[gene_id] = gene
                return list(gene_dict.values())

# ==================== textprocesstext ====================

def process_taxon(taxon: str, sample_id: str = None) -> pd.DataFrame:
    """
    processtextmicrobesallsample(ortextsample)
    text:
    taxon: microbetext
    sample_id: textsampleID(optional, text)
    return:
    DataFrametextgeneinformation
    """
    print(f"\n{'='*70}")
    print(f"processmicrobe: {taxon}")
    print(f"{'='*70}")
    # 1. loadannotationfile
    print("\n[1/6] loadgeneannotation...")
    gff_file = os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz")
    gtf_file = os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz")
    import glob
    gff_files = glob.glob(gff_file)
    gtf_files = glob.glob(gtf_file)
    annotations = {}
    file_type = None
    if gff_files:
        print(f" textGFF: {os.path.basename(gff_files[0])}")
        annotations = load_annotations(gff_files[0], 'gff')
        file_type = 'gff'
    elif gtf_files:
        print(f" textGTF: {os.path.basename(gtf_files[0])}")
        annotations = load_annotations(gtf_files[0], 'gtf')
        file_type = 'gtf'
    else:
        print(f" error: not foundannotationfile")
        return pd.DataFrame()
    # deduplicategene
    all_genes = []
    for chrom, genes in annotations.items():
        all_genes.extend(genes)
        unique_genes = deduplicate_genes(all_genes)
        print(f" loadtext {len(unique_genes)} genes")
        # 2. calculateNormalsampletext
        print("\n[2/6] calculateNormaltext(fromflagstat)...")
        normal_total_size = get_normal_total_library_size()
        if normal_total_size == 0:
            print(f" error: notextNormaltext")
            return pd.DataFrame()
        # normalizeto1M reads
        normal_norm_factor = 1e6 / normal_total_size
        print(f" Normalnormalization factor: {normal_norm_factor:.8f}")
        # 3. loadNormalsamplecoverage(after normalization)
        print("\n[3/6] loadNormalcoverage(normalize)...")
        normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'coverage_Normal.bedgraph')
        normal_coverage = load_bedgraph(normal_bedgraph, normal_norm_factor)
        print(f" Normalcoveragetext: {sum(len(v) for v in normal_coverage.values())}")
        # 4. findTargetsample
        print("\n[4/6] findTargetsample...")
        target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
        if not os.path.exists(target_dir):
            print(f" error: Targetdirectorytextexists: {target_dir}")
            return pd.DataFrame()
        bedgraph_files = [f for f in os.listdir(target_dir)             if f.endswith('_rna_output.pathseq.bedgraph')]
        if sample_id:
            # onlyprocesstextsample
            bedgraph_files = [f for f in bedgraph_files if f.startswith(f"{sample_id}_")]
            print(f" textsample: {sample_id}")
            print(f" found {len(bedgraph_files)} Targetsample")
            # 5. processtextsamples
            print("\n[5/6] calculategenecoverageandtext(after normalization)...")
            all_results = []
            for idx, bedgraph_file in enumerate(bedgraph_files, 1):
                sample_name = bedgraph_file.replace('_rna_output.pathseq.bedgraph', '')
                print(f"\n [{idx}/{len(bedgraph_files)}] sample: {sample_name}")
                # fromflagstattextTargetsampletext
                target_flagstat = os.path.join(config.TARGET_FLAGSTAT_DIR,                     f"{sample_name}_rna_hg38_align_sort.bam.flagstat.txt")
                target_library_size = get_library_size_from_flagstat(target_flagstat)
                if target_library_size == 0:
                    print(f" warning: notextTargettext, skiptextsample")
                    continue
                print(f" Targettext: {target_library_size:,}")
                target_norm_factor = 1e6 / target_library_size
                print(f" Targetnormalization factor: {target_norm_factor:.8f}")
                print(f" text (Target/Normal): {target_library_size/normal_total_size:.4f}")
                # loadTargetcoverage(after normalization)
                target_bedgraph = os.path.join(target_dir, bedgraph_file)
                target_coverage = load_bedgraph(target_bedgraph, target_norm_factor)
                sample_results = []
                for gene in unique_genes:
                    seqid = gene['seqid']
                    # calculateTargetcoverage
                    target_stats = calculate_gene_coverage(
                        gene['start'], gene['end'],
                        target_coverage.get(seqid, [])
)
# calculateNormalcoverage
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(seqid, [])
)
# filtertext
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue # insufficient target coverage
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue # target coverage too low
    # calculatefold change (text)
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue # insufficient fold change
    # optional: filterNormalcoveragetextgene
    if config.MAX_NORMAL_COVERAGE and normal_stats['mean_coverage'] > config.MAX_NORMAL_COVERAGE:
        continue
    # saveresults
    result = {
        'sample': sample_name,
        'taxon': taxon,
        'seqid': seqid,
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        # Targetcoveragestatistics (normalizeto1Mtext)
        'target_norm_cov': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        # Normalcoveragestatistics (normalizeto1Mtext)
        'normal_norm_cov': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        # text
        'fold_change': round(fold_change, 2),
        'coverage_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    }
    if file_type == 'gtf':
        result['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        result['locus_tag'] = gene.get('locus_tag', 'N/A')
        sample_results.append(result)
        print(f" found {len(sample_results)} textgene")
        all_results.extend(sample_results)
        # 6. textresults
        print(f"\n[6/6] text: {len(all_results)} textgene")
        if not all_results:
            return pd.DataFrame()
        df = pd.DataFrame(all_results)
        df = df.sort_values(['sample', 'fold_change'], ascending=[True, False])
        return df

def extract_gene_sequences(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """
    fromtextgenetextextractgenesequence
    return: {gene_key: sequence}
    """
    print(f"\nextractgenesequence...")
    # loadtextgenetext
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        print(f" warning: textgenetextexists: {ref_file}")
        return {}
    genome_seqs = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        print(f" loadtext {len(genome_seqs)} textsequence")
        # extractgenesequence
        gene_sequences = {}
        for _, row in df.iterrows():
            seqid = row['seqid']
            start = row['gene_start']
            end = row['gene_end']
            if seqid not in genome_seqs:
                continue
            sequence = genome_seqs[seqid][start-1:end]
            gene_key = f"{row['sample']}|{seqid}_{start}_{end}"
            gene_sequences[gene_key] = sequence
            print(f" extracttext {len(gene_sequences)} genessequence")
            return gene_sequences

def save_results(df: pd.DataFrame, taxon: str, sample_id: str = None):
    """saveresults"""
    if df.empty:
        print("\ntextwithresultstextsave")
        return
    # Create output directory
    output_dir = os.path.join(config.OUTPUT_DIR, 'reports')
    os.makedirs(output_dir, exist_ok=True)
    # saveCSV
    if sample_id:
        csv_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.csv")
    else:
        csv_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.csv")
        df.to_csv(csv_file, index=False)
        print(f"\nOK resultssaved: {csv_file}")
        # extracttextsavesequence
        gene_sequences = extract_gene_sequences(df, taxon)
        if gene_sequences:
            if sample_id:
                fna_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.fna")
            else:
                fna_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.fna")
                with open(fna_file, 'w') as f:
                    for gene_key, sequence in gene_sequences.items():
                        parts = gene_key.split('|')
                        sample = parts[0]
                        coords = parts[1]
                        # fromdfintextgeneinformation
                        mask = (df['sample'] == sample) & (df['seqid'] + '_' + df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str) == coords)
                        if mask.any():
                            gene_info = df[mask].iloc[0]
                            header = f">{coords} {gene_info['gene_name']} | {gene_info['product']} | FC={gene_info['fold_change']}"
                        else:
                            header = f">{coords}"
                            f.write(header + '\n')
                            # textrows60text
                            for i in range(0, len(sequence), 60):
                                f.write(sequence[i:i+60] + '\n')
                                print(f"OK sequencesaved: {fna_file}")
                                # textstatisticstext
                                print(f"\n{'='*70}")
                                print("resultstext")
                                print(f"{'='*70}")
                                print(f"microbe: {taxon}")
                                print(f"sample count: {df['sample'].nunique()}")
                                print(f"textgenetext: {len(df)}")
                                print(f"\nFold Changetext:")
                                print(df['fold_change'].describe())
                                print(f"\nTargetnormalizecoveragetext:")
                                print(df['target_norm_cov'].describe())
                                print(f"\nNormalnormalizecoveragetext:")
                                print(df['normal_norm_cov'].describe())
                                print(f"\ntextsamplestatistics:")
                                print(df.groupby('sample').size().sort_values(ascending=False))

# ==================== text ====================

def main():
    """text"""
    print("=" * 70)
    print("MTRgenefilter - textanalyzetext (normalizecoverage)")
    print("=" * 70)
    print("\ntext:")
    print(f" minimum gene length: {config.MIN_GENE_LENGTH} bp")
    print(f" textTargetcoverage: {config.MIN_TARGET_COVERAGE} (normalize)")
    print(f" textFold Change: {config.MIN_FOLD_CHANGE}")
    print(f" genecoveragetext: {config.MIN_GENE_COVERAGE_FRACTION}")
    if config.MAX_NORMAL_COVERAGE:
        print(f" textNormalcoverage: {config.MAX_NORMAL_COVERAGE} (normalize)")
        print("\nnormalizetext:")
        print(" - fromflagstatfilereadtext")
        print(" - Target: textsampletext (paired in sequencing)")
        print(" - Normal: 89sampletextand")
        print(" - normalization factor = 1,000,000 / text")
        print(" - normalizecoverage = textcoverage × normalization factor")
        print(" - textallsampletexttotext(1M reads)")
        # text: onlyprocessB7sample
        print("\n" + "="*70)
        print("text: onlyprocessB7sample")
        print("="*70)
        # readmicrobe list
        taxon_df = pd.read_csv(os.path.join(config.BASE_DIR, config.TAXON_LIST))
        taxons = taxon_df['Taxon'].tolist()
        print(f"\nshared {len(taxons)} microbes, text...")
        # textmicrobesB7sample
        test_taxon = taxons[0]
        start_time = time.time()
        df = process_taxon(test_taxon, sample_id='B7')
        if not df.empty:
            save_results(df, test_taxon, sample_id='B7')
            elapsed = time.time() - start_time
            print(f"\ntext: {elapsed:.2f} text")
            print("\n" + "="*70)
            print("textcompleted!textcheckresults, textreasonableaftertextprocesstextdata")
            print("="*70)

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - textanalyzetext
====================================

text: 1. textgenetextcalculatecoverage
2. **normalizetotextmicrobereadstext** (RPM for microbial reads)
3. textfold changetext
4. text0/1text

normalizetext (correcttext): - text: onlytexttotextmicrobereadstextrowsnormalize
- Target: from {sample}_rna_output.pathseq_sorted.bam.txt statisticsrowstext
- Normal: from merged_Normal_sorted.bam.txt statisticsrowstext
- normalization factor = 1,000,000 / microbemapped readstext
- normalizecoverage = textcoverage × normalization factor

fortext(flagstat)?
- flagstatstatisticstextgenetextresults(hg38)
- textgenetextreads, textmicrobenotext
- textnormalizetextmicrobetext
- textmicrobetextmapped readstext

fortextRPMtextRPKM?
- bedgraphtextcoverage
- calculatetextaveragecoverage, textgenetext
- onlytextnormalizetextcompare
- text"reads per million microbial reads"

text: - microbegenetext, textwithtext
- textgene(targettextnormal)
- textandgene cluster
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import time

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    # path
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    # Targetsamplepath
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    # Normalsamplepath
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    # outputpath
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50 # minimum gene length(bp)
    MIN_TARGET_COVERAGE: float = 0.5 # minimum target average coverage(after normalization)
    MIN_FOLD_CHANGE: float = 1.5 # textfold change (target/normal)
    MAX_NORMAL_COVERAGE: float = None # maximum normal average coverage(None=no limit)
    # coveragecalculatetext
    MIN_GENE_COVERAGE_FRACTION: float = 0.3 # at least 30% of the gene region has coverage
    def __post_init__(self):
        """buildtextpath"""
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'NORMAL_COVERAGE_DIR', 'OUTPUT_DIR']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== Utility functions ====================

def get_microbial_mapped_reads(bam_txt_file: str) -> int:
    """
    from.bam.txtfilestatisticstexttotextmicrobetextreadstext
    text, textfor:     1. filetexttotextmicroberesults
    2. rowstext = mapped readstext
    3. flagstattextgenetextstatistics, text
    """
    if not os.path.exists(bam_txt_file):
        return 0
    read_count = 0
    with open(bam_txt_file, 'r') as f:
        for line in f:
            if not line.startswith('@'):
                read_count += 1
                return read_count

def get_normal_total_microbial_reads(taxon: str) -> int:
    """
    calculateNormalsampletexttotextmicrobetextreadstext
    textmerged_Normal_sorted.bam.txt
    """
    bam_txt = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,         'merged_Normal_sorted.bam.txt')
    total_reads = get_microbial_mapped_reads(bam_txt)
    if total_reads > 0:
        print(f" Normaltextto{taxon}reads: {total_reads:,}")
    else:
        print(f" warning: notextreadNormal.bam.txt")
        return total_reads

def load_bedgraph(bedgraph_file: str, normalize_factor: float = 1.0) -> Dict[str, List[Tuple[int, int, float]]]:
    """
    loadbedgraphfile, textrowsnormalize
    text:
    bedgraph_file: bedgraphfilepath
    normalize_factor: normalization factor (targettext / actualtext)
    return: {chromosome: [(start, end, normalized_coverage), ...]}
    """
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        print(f" warning: bedgraphfiletextexists: {bedgraph_file}")
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                coverage = float(parts[3]) * normalize_factor # normalize
                regions[chrom].append((start, end, coverage))
                return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
"""
calculategenetextcoveragestatistics
return:
{
'mean_coverage': averagecoverage,
'covered_bases': withcoveragetext,
'total_bases': genetext,
'coverage_fraction': coveragetext
}
"""
gene_length = gene_end - gene_start
# texteachtextcoverage
coverage_array = np.zeros(gene_length, dtype=float)
# textcoverage
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    # calculatetext
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        # textfortextgenetext
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        # calculatestatisticstext
        covered_bases = np.sum(coverage_array > 0)
        mean_coverage = np.mean(coverage_array)
        coverage_fraction = covered_bases / gene_length if gene_length > 0 else 0
        return {
        'mean_coverage': mean_coverage,
        'covered_bases': int(covered_bases),
        'total_bases': gene_length,
        'coverage_fraction': coverage_fraction
    }

# ==================== annotationfiletext ====================

def parse_gff_line(line: str) -> Optional[Dict]:
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'source': fields[1],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string: str) -> Dict[str, str]:
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string: str) -> Dict[str, str]:
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotations(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """
    loadannotationfile
    return: {chromosome: [gene_info, ...]}
    """
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    parse_attrs = parse_gtf_attributes if file_type == 'gtf' else parse_gff_attributes
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if not record:
                continue
            # onlytextgenetext(orCDStext)
            if record['type'] not in ['gene', 'CDS', 'mRNA', 'transcript']:
                continue
            attrs = parse_attrs(record['attributes'])
            gene_info = {
                'seqid': record['seqid'],
                'type': record['type'],
                'start': record['start'],
                'end': record['end'],
                'strand': record['strand'],
                'length': record['end'] - record['start'],
            }
            # extractgeneIDandtext
            if file_type == 'gtf':
                gene_info['gene_id'] = attrs.get('gene_id', 'N/A')
                gene_info['gene_name'] = attrs.get('gene_name', attrs.get('gene_id', 'N/A'))
                gene_info['gene_biotype'] = attrs.get('gene_biotype', 'N/A')
                gene_info['product'] = attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            else: # gff
                gene_info['gene_id'] = attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A')))
                gene_info['gene_name'] = attrs.get('Name', attrs.get('gene', 'N/A'))
                gene_info['product'] = attrs.get('product', 'N/A')
                gene_info['locus_tag'] = attrs.get('locus_tag', 'N/A')
                annotations[record['seqid']].append(gene_info)
                return annotations

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    """
    deduplicategene(textgenetextfeature, text)
    """
    gene_dict = {}
    for gene in genes:
        # filtertextgene
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict:
            gene_dict[gene_id] = gene
        else:
            # textfeature
            if gene['length'] > gene_dict[gene_id]['length']:
                gene_dict[gene_id] = gene
                return list(gene_dict.values())

# ==================== textprocesstext ====================

def process_taxon(taxon: str, sample_id: str = None) -> pd.DataFrame:
    """
    processtextmicrobesallsample(ortextsample)
    text:
    taxon: microbetext
    sample_id: textsampleID(optional, text)
    return:
    DataFrametextgeneinformation
    """
    print(f"\n{'='*70}")
    print(f"processmicrobe: {taxon}")
    print(f"{'='*70}")
    # 1. loadannotationfile
    print("\n[1/6] loadgeneannotation...")
    gff_file = os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz")
    gtf_file = os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz")
    import glob
    gff_files = glob.glob(gff_file)
    gtf_files = glob.glob(gtf_file)
    annotations = {}
    file_type = None
    if gff_files:
        print(f" textGFF: {os.path.basename(gff_files[0])}")
        annotations = load_annotations(gff_files[0], 'gff')
        file_type = 'gff'
    elif gtf_files:
        print(f" textGTF: {os.path.basename(gtf_files[0])}")
        annotations = load_annotations(gtf_files[0], 'gtf')
        file_type = 'gtf'
    else:
        print(f" error: not foundannotationfile")
        return pd.DataFrame()
    # deduplicategene
    all_genes = []
    for chrom, genes in annotations.items():
        all_genes.extend(genes)
        unique_genes = deduplicate_genes(all_genes)
        print(f" loadtext {len(unique_genes)} genes")
        # 2. calculateNormalsampletexttotextmicrobereadstext
        print("\n[2/6] calculateNormaltexttotextmicrobereadstext...")
        normal_microbial_reads = get_normal_total_microbial_reads(taxon)
        if normal_microbial_reads == 0:
            print(f" error: notextNormalmicrobereadstext")
            return pd.DataFrame()
        # normalizeto1M reads
        normal_norm_factor = 1e6 / normal_microbial_reads
        print(f" Normalnormalization factor: {normal_norm_factor:.6f} (RPM)")
        # 3. loadNormalsamplecoverage(after normalization)
        print("\n[3/6] loadNormalcoverage(normalizetoRPM)...")
        normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'coverage_Normal.bedgraph')
        normal_coverage = load_bedgraph(normal_bedgraph, normal_norm_factor)
        print(f" Normalcoveragetext: {sum(len(v) for v in normal_coverage.values())}")
        # 4. findTargetsample
        print("\n[4/6] findTargetsample...")
        target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
        if not os.path.exists(target_dir):
            print(f" error: Targetdirectorytextexists: {target_dir}")
            return pd.DataFrame()
        bedgraph_files = [f for f in os.listdir(target_dir)             if f.endswith('_rna_output.pathseq.bedgraph')]
        if sample_id:
            # onlyprocesstextsample
            bedgraph_files = [f for f in bedgraph_files if f.startswith(f"{sample_id}_")]
            print(f" textsample: {sample_id}")
            print(f" found {len(bedgraph_files)} Targetsample")
            # 5. processtextsamples
            print("\n[5/6] calculategenecoverageandtext(after normalization)...")
            all_results = []
            for idx, bedgraph_file in enumerate(bedgraph_files, 1):
                sample_name = bedgraph_file.replace('_rna_output.pathseq.bedgraph', '')
                print(f"\n [{idx}/{len(bedgraph_files)}] sample: {sample_name}")
                # from.bam.txttextTargetsampletexttotextmicrobereadstext
                target_bam_txt = os.path.join(target_dir,                     f"{sample_name}_rna_output.pathseq_sorted.bam.txt")
                target_microbial_reads = get_microbial_mapped_reads(target_bam_txt)
                if target_microbial_reads == 0:
                    print(f" warning: notextTargetmicrobereadstext, skiptextsample")
                    continue
                print(f" Targettextto{taxon}reads: {target_microbial_reads:,}")
                target_norm_factor = 1e6 / target_microbial_reads
                print(f" Targetnormalization factor: {target_norm_factor:.6f} (RPM)")
                print(f" microbereadstext (Target/Normal): {target_microbial_reads/normal_microbial_reads:.6f}")
                # loadTargetcoverage(after normalization)
                target_bedgraph = os.path.join(target_dir, bedgraph_file)
                target_coverage = load_bedgraph(target_bedgraph, target_norm_factor)
                sample_results = []
                for gene in unique_genes:
                    seqid = gene['seqid']
                    # calculateTargetcoverage
                    target_stats = calculate_gene_coverage(
                        gene['start'], gene['end'],
                        target_coverage.get(seqid, [])
)
# calculateNormalcoverage
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(seqid, [])
)
# filtertext
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue # insufficient target coverage
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue # target coverage too low
    # calculatefold change (text)
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue # insufficient fold change
    # optional: filterNormalcoveragetextgene
    if config.MAX_NORMAL_COVERAGE and normal_stats['mean_coverage'] > config.MAX_NORMAL_COVERAGE:
        continue
    # saveresults
    result = {
        'sample': sample_name,
        'taxon': taxon,
        'seqid': seqid,
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        # Targetcoveragestatistics (RPM: normalizeto1Mmicrobereads)
        'target_rpm': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        # Normalcoveragestatistics (RPM: normalizeto1Mmicrobereads)
        'normal_rpm': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        # text
        'fold_change': round(fold_change, 2),
        'coverage_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    }
    if file_type == 'gtf':
        result['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        result['locus_tag'] = gene.get('locus_tag', 'N/A')
        sample_results.append(result)
        print(f" found {len(sample_results)} textgene")
        all_results.extend(sample_results)
        # 6. textresults
        print(f"\n[6/6] text: {len(all_results)} textgene")
        if not all_results:
            return pd.DataFrame()
        df = pd.DataFrame(all_results)
        df = df.sort_values(['sample', 'fold_change'], ascending=[True, False])
        return df

def extract_gene_sequences(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """
    fromtextgenetextextractgenesequence
    return: {gene_key: sequence}
    """
    print(f"\nextractgenesequence...")
    # loadtextgenetext
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        print(f" warning: textgenetextexists: {ref_file}")
        return {}
    genome_seqs = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        print(f" loadtext {len(genome_seqs)} textsequence")
        # extractgenesequence
        gene_sequences = {}
        for _, row in df.iterrows():
            seqid = row['seqid']
            start = row['gene_start']
            end = row['gene_end']
            if seqid not in genome_seqs:
                continue
            sequence = genome_seqs[seqid][start-1:end]
            gene_key = f"{row['sample']}|{seqid}_{start}_{end}"
            gene_sequences[gene_key] = sequence
            print(f" extracttext {len(gene_sequences)} genessequence")
            return gene_sequences

def save_results(df: pd.DataFrame, taxon: str, sample_id: str = None):
    """saveresults"""
    if df.empty:
        print("\ntextwithresultstextsave")
        return
    # Create output directory
    output_dir = os.path.join(config.OUTPUT_DIR, 'reports')
    os.makedirs(output_dir, exist_ok=True)
    # saveCSV
    if sample_id:
        csv_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.csv")
    else:
        csv_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.csv")
        df.to_csv(csv_file, index=False)
        print(f"\nOK resultssaved: {csv_file}")
        # extracttextsavesequence
        gene_sequences = extract_gene_sequences(df, taxon)
        if gene_sequences:
            if sample_id:
                fna_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes.fna")
            else:
                fna_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples.fna")
                with open(fna_file, 'w') as f:
                    for gene_key, sequence in gene_sequences.items():
                        parts = gene_key.split('|')
                        sample = parts[0]
                        coords = parts[1]
                        # fromdfintextgeneinformation
                        mask = (df['sample'] == sample) & (df['seqid'] + '_' + df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str) == coords)
                        if mask.any():
                            gene_info = df[mask].iloc[0]
                            header = f">{coords} {gene_info['gene_name']} | {gene_info['product']} | FC={gene_info['fold_change']}"
                        else:
                            header = f">{coords}"
                            f.write(header + '\n')
                            # textrows60text
                            for i in range(0, len(sequence), 60):
                                f.write(sequence[i:i+60] + '\n')
                                print(f"OK sequencesaved: {fna_file}")
                                # textstatisticstext
                                print(f"\n{'='*70}")
                                print("resultstext")
                                print(f"{'='*70}")
                                print(f"microbe: {taxon}")
                                print(f"sample count: {df['sample'].nunique()}")
                                print(f"textgenetext: {len(df)}")
                                print(f"\nFold Changetext:")
                                print(df['fold_change'].describe())
                                print(f"\nTarget RPMtext:")
                                print(df['target_rpm'].describe())
                                print(f"\nNormal RPMtext:")
                                print(df['normal_rpm'].describe())
                                print(f"\ntextsamplestatistics:")
                                print(df.groupby('sample').size().sort_values(ascending=False))

# ==================== text ====================

def main():
    """text"""
    print("=" * 70)
    print("MTRgenefilter - RPMnormalizetext(microbereads)")
    print("=" * 70)
    print("\ntext:")
    print(f" minimum gene length: {config.MIN_GENE_LENGTH} bp")
    print(f" textTargetcoverage: {config.MIN_TARGET_COVERAGE} (RPM)")
    print(f" textFold Change: {config.MIN_FOLD_CHANGE}")
    print(f" genecoveragetext: {config.MIN_GENE_COVERAGE_FRACTION}")
    if config.MAX_NORMAL_COVERAGE:
        print(f" textNormalcoverage: {config.MAX_NORMAL_COVERAGE} (RPM)")
        print("\nnormalizetext:")
        print(" - onlytexttotextmicrobereadstextrowsnormalize")
        print(" - Target: from .bam.txt statisticsmicrobemapped reads")
        print(" - Normal: from merged_Normal.bam.txt statistics")
        print(" - RPM = (coverage / microbereads) × 1,000,000")
        print(" - textflagstat(textgenetextstatistics)")
        # text: onlyprocessB7sample
        print("\n" + "="*70)
        print("text: onlyprocessB7sample")
        print("="*70)
        # readmicrobe list
        taxon_df = pd.read_csv(os.path.join(config.BASE_DIR, config.TAXON_LIST))
        taxons = taxon_df['Taxon'].tolist()
        print(f"\nshared {len(taxons)} microbes, text...")
        # textmicrobesB7sample
        test_taxon = taxons[0]
        start_time = time.time()
        df = process_taxon(test_taxon, sample_id='B7')
        if not df.empty:
            save_results(df, test_taxon, sample_id='B7')
            elapsed = time.time() - start_time
            print(f"\ntext: {elapsed:.2f} text")
            print("\n" + "="*70)
            print("textcompleted!textcheckresults, textreasonableaftertextprocesstextdata")
            print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - textanalyzetext
====================================

text: 1. textgenetextcalculatecoverage
2. **normalizetotextmicrobereadstext** (RPM for microbial reads)
3. textfold changetext
4. text0/1text

normalizetext (correcttext): - text: onlytexttotextmicrobereadstextrowsnormalize
- Target: from {sample}_rna_output.pathseq_sorted.bam.txt statisticsrowstext
- Normal: from merged_Normal_sorted.bam.txt statisticsrowstext
- normalization factor = 1,000,000 / microbemapped readstext
- normalizecoverage = textcoverage × normalization factor

fortext(flagstat)?
- flagstatstatisticstextgenetextresults(hg38)
- textgenetextreads, textmicrobenotext
- textnormalizetextmicrobetext
- textmicrobetextmapped readstext

fortextRPMtextRPKM?
- bedgraphtextcoverage
- calculatetextaveragecoverage, textgenetext
- onlytextnormalizetextcompare
- text"reads per million microbial reads"

text: - microbegenetext, textwithtext
- textgene(targettextnormal)
- textandgene cluster
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import time

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    # path
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    # Targetsamplepath
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    # Normalsamplepath
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    # outputpath
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50 # minimum gene length(bp)
    MIN_TARGET_COVERAGE: float = 0.5 # minimum target average coverage(after normalization)
    MIN_FOLD_CHANGE: float = 1.5 # textfold change (target/normal)
    MAX_NORMAL_COVERAGE: float = None # maximum normal average coverage(None=no limit)
    # coveragecalculatetext
    MIN_GENE_COVERAGE_FRACTION: float = 0.3 # at least 30% of the gene region has coverage
    def __post_init__(self):
        """buildtextpath"""
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'NORMAL_COVERAGE_DIR', 'OUTPUT_DIR']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== Utility functions ====================

def get_microbial_mapped_reads(bam_txt_file: str) -> int:
    """
    from.bam.txtfilestatisticstexttotextmicrobetextreadstext
    text, textfor:     1. filetexttotextmicroberesults
    2. rowstext = mapped readstext
    3. flagstattextgenetextstatistics, text
    """
    if not os.path.exists(bam_txt_file):
        return 0
    read_count = 0
    with open(bam_txt_file, 'r') as f:
        for line in f:
            if not line.startswith('@'):
                read_count += 1
                return read_count

def get_normal_total_microbial_reads(taxon: str) -> int:
    """
    calculateNormalsampletexttotextmicrobetextreadstext
    textmerged_Normal_sorted.bam.txt
    """
    bam_txt = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,         'merged_Normal_sorted.bam.txt')
    total_reads = get_microbial_mapped_reads(bam_txt)
    if total_reads > 0:
        print(f" Normaltextto{taxon}reads: {total_reads:,}")
    else:
        print(f" warning: notextreadNormal.bam.txt")
        return total_reads

def load_bedgraph(bedgraph_file: str, normalize_factor: float = 1.0) -> Dict[str, List[Tuple[int, int, float]]]:
    """
    loadbedgraphfile, textrowsnormalize
    text:
    bedgraph_file: bedgraphfilepath
    normalize_factor: normalization factor (targettext / actualtext)
    return: {chromosome: [(start, end, normalized_coverage), ...]}
    """
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        print(f" warning: bedgraphfiletextexists: {bedgraph_file}")
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                coverage = float(parts[3]) * normalize_factor # normalize
                regions[chrom].append((start, end, coverage))
                return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
"""
calculategenetextcoveragestatistics
return:
{
'mean_coverage': averagecoverage,
'covered_bases': withcoveragetext,
'total_bases': genetext,
'coverage_fraction': coveragetext
}
"""
gene_length = gene_end - gene_start
# texteachtextcoverage
coverage_array = np.zeros(gene_length, dtype=float)
# textcoverage
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    # calculatetext
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        # textfortextgenetext
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        # calculatestatisticstext
        covered_bases = np.sum(coverage_array > 0)
        mean_coverage = np.mean(coverage_array)
        coverage_fraction = covered_bases / gene_length if gene_length > 0 else 0
        return {
        'mean_coverage': mean_coverage,
        'covered_bases': int(covered_bases),
        'total_bases': gene_length,
        'coverage_fraction': coverage_fraction
    }

# ==================== annotationfiletext ====================

def parse_gff_line(line: str) -> Optional[Dict]:
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'source': fields[1],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string: str) -> Dict[str, str]:
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string: str) -> Dict[str, str]:
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotations(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """
    loadannotationfile
    return: {chromosome: [gene_info, ...]}
    """
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    parse_attrs = parse_gtf_attributes if file_type == 'gtf' else parse_gff_attributes
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if not record:
                continue
            # onlytextgenetext(orCDStext)
            if record['type'] not in ['gene', 'CDS', 'mRNA', 'transcript']:
                continue
            attrs = parse_attrs(record['attributes'])
            gene_info = {
                'seqid': record['seqid'],
                'type': record['type'],
                'start': record['start'],
                'end': record['end'],
                'strand': record['strand'],
                'length': record['end'] - record['start'],
            }
            # extractgeneIDandtext
            if file_type == 'gtf':
                gene_info['gene_id'] = attrs.get('gene_id', 'N/A')
                gene_info['gene_name'] = attrs.get('gene_name', attrs.get('gene_id', 'N/A'))
                gene_info['gene_biotype'] = attrs.get('gene_biotype', 'N/A')
                gene_info['product'] = attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            else: # gff
                gene_info['gene_id'] = attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A')))
                gene_info['gene_name'] = attrs.get('Name', attrs.get('gene', 'N/A'))
                gene_info['product'] = attrs.get('product', 'N/A')
                gene_info['locus_tag'] = attrs.get('locus_tag', 'N/A')
                annotations[record['seqid']].append(gene_info)
                return annotations

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    """
    deduplicategene(textgenetextfeature, text)
    """
    gene_dict = {}
    for gene in genes:
        # filtertextgene
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict:
            gene_dict[gene_id] = gene
        else:
            # textfeature
            if gene['length'] > gene_dict[gene_id]['length']:
                gene_dict[gene_id] = gene
                return list(gene_dict.values())

# ==================== textprocesstext ====================

def process_taxon(taxon: str, sample_id: str = None) -> pd.DataFrame:
    """
    processtextmicrobesallsample(ortextsample)
    text:
    taxon: microbetext
    sample_id: textsampleID(optional, text)
    return:
    DataFrametextgeneinformation
    """
    print(f"\n{'='*70}")
    print(f"processmicrobe: {taxon}")
    print(f"{'='*70}")
    # 1. loadannotationfile
    print("\n[1/6] loadgeneannotation...")
    gff_file = os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz")
    gtf_file = os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz")
    import glob
    gff_files = glob.glob(gff_file)
    gtf_files = glob.glob(gtf_file)
    annotations = {}
    file_type = None
    if gff_files:
        print(f" textGFF: {os.path.basename(gff_files[0])}")
        annotations = load_annotations(gff_files[0], 'gff')
        file_type = 'gff'
    elif gtf_files:
        print(f" textGTF: {os.path.basename(gtf_files[0])}")
        annotations = load_annotations(gtf_files[0], 'gtf')
        file_type = 'gtf'
    else:
        print(f" error: not foundannotationfile")
        return pd.DataFrame()
    # deduplicategene
    all_genes = []
    for chrom, genes in annotations.items():
        all_genes.extend(genes)
        unique_genes = deduplicate_genes(all_genes)
        print(f" loadtext {len(unique_genes)} genes")
        # 2. calculateNormalsampletexttotextmicrobereadstext
        print("\n[2/6] calculateNormaltexttotextmicrobereadstext...")
        normal_microbial_reads = get_normal_total_microbial_reads(taxon)
        if normal_microbial_reads == 0:
            print(f" error: notextNormalmicrobereadstext")
            return pd.DataFrame()
        # normalizeto1M reads
        normal_norm_factor = 1e6 / normal_microbial_reads
        print(f" Normalnormalization factor: {normal_norm_factor:.6f} (RPM)")
        # 3. loadNormalsamplecoverage(after normalization)
        print("\n[3/6] loadNormalcoverage(normalizetoRPM)...")
        normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'coverage_Normal.bedgraph')
        normal_coverage = load_bedgraph(normal_bedgraph, normal_norm_factor)
        print(f" Normalcoveragetext: {sum(len(v) for v in normal_coverage.values())}")
        # 4. findTargetsample
        print("\n[4/6] findTargetsample...")
        target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
        if not os.path.exists(target_dir):
            print(f" error: Targetdirectorytextexists: {target_dir}")
            return pd.DataFrame()
        bedgraph_files = [f for f in os.listdir(target_dir)             if f.endswith('_rna_output.pathseq.bedgraph')]
        if sample_id:
            # onlyprocesstextsample
            bedgraph_files = [f for f in bedgraph_files if f.startswith(f"{sample_id}_")]
            print(f" textsample: {sample_id}")
            print(f" found {len(bedgraph_files)} Targetsample")
            # 5. processtextsamples
            print("\n[5/6] calculategenecoverageandtext(after normalization)...")
            all_results = []
            for idx, bedgraph_file in enumerate(bedgraph_files, 1):
                sample_name = bedgraph_file.replace('_rna_output.pathseq.bedgraph', '')
                print(f"\n [{idx}/{len(bedgraph_files)}] sample: {sample_name}")
                # from.bam.txttextTargetsampletexttotextmicrobereadstext
                target_bam_txt = os.path.join(target_dir,                     f"{sample_name}_rna_output.pathseq_sorted.bam.txt")
                target_microbial_reads = get_microbial_mapped_reads(target_bam_txt)
                if target_microbial_reads == 0:
                    print(f" warning: notextTargetmicrobereadstext, skiptextsample")
                    continue
                print(f" Targettextto{taxon}reads: {target_microbial_reads:,}")
                target_norm_factor = 1e6 / target_microbial_reads
                print(f" Targetnormalization factor: {target_norm_factor:.6f} (RPM)")
                print(f" microbereadstext (Target/Normal): {target_microbial_reads/normal_microbial_reads:.6f}")
                # loadTargetcoverage(after normalization)
                target_bedgraph = os.path.join(target_dir, bedgraph_file)
                target_coverage = load_bedgraph(target_bedgraph, target_norm_factor)
                sample_results = []
                for gene in unique_genes:
                    seqid = gene['seqid']
                    # calculateTargetcoverage
                    target_stats = calculate_gene_coverage(
                        gene['start'], gene['end'],
                        target_coverage.get(seqid, [])
)
# calculateNormalcoverage
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(seqid, [])
)
# filtertext
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue # insufficient target coverage
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue # target coverage too low
    # calculatefold change (text)
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue # insufficient fold change
    # optional: filterNormalcoveragetextgene
    if config.MAX_NORMAL_COVERAGE and normal_stats['mean_coverage'] > config.MAX_NORMAL_COVERAGE:
        continue
    # saveresults
    result = {
        'sample': sample_name,
        'taxon': taxon,
        'seqid': seqid,
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        # Targetcoveragestatistics (RPM: normalizeto1Mmicrobereads)
        'target_rpm': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        # Normalcoveragestatistics (RPM: normalizeto1Mmicrobereads)
        'normal_rpm': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        # text
        'fold_change': round(fold_change, 2),
        'coverage_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    }
    if file_type == 'gtf':
        result['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        result['locus_tag'] = gene.get('locus_tag', 'N/A')
        sample_results.append(result)
        print(f" found {len(sample_results)} textgene")
        all_results.extend(sample_results)
        # 6. textresults
        print(f"\n[6/6] text: {len(all_results)} textgene")
        if not all_results:
            return pd.DataFrame(), None
        df = pd.DataFrame(all_results)
        df = df.sort_values(['sample', 'fold_change'], ascending=[True, False])
        return df, file_type # returnfile_type

def extract_gene_sequences(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """
    fromtextgenetextextractgenesequence
    return: {gene_key: sequence}
    """
    print(f"\nextractgenesequence...")
    # loadtextgenetext
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        print(f" warning: textgenetextexists: {ref_file}")
        return {}
    genome_seqs = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        print(f" loadtext {len(genome_seqs)} textsequence")
        # extractgenesequence
        gene_sequences = {}
        for _, row in df.iterrows():
            seqid = row['seqid']
            start = row['gene_start']
            end = row['gene_end']
            if seqid not in genome_seqs:
                continue
            sequence = genome_seqs[seqid][start-1:end]
            gene_key = f"{row['sample']}|{seqid}_{start}_{end}"
            gene_sequences[gene_key] = sequence
            print(f" extracttext {len(gene_sequences)} genessequence")
            return gene_sequences

def save_results(df: pd.DataFrame, taxon: str, file_type: str, sample_id: str = None):
    """saveresults
    text:
    df: resultsDataFrame
    taxon: microbetext
    file_type: annotationfiletext ('gff' or 'gtf')
    sample_id: sampleID
    """
    if df.empty:
        print("\ntextwithresultstextsave")
        return
    # Create output directory
    output_dir = os.path.join(config.OUTPUT_DIR, 'reports')
    os.makedirs(output_dir, exist_ok=True)
    # saveCSV (filenametextannotationtext)
    if sample_id:
        csv_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes_{file_type.upper()}.csv")
    else:
        csv_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples_{file_type.upper()}.csv")
        df.to_csv(csv_file, index=False)
        # texttoCSVtextdirectory
        method_file = os.path.join(output_dir, "METHODS_README.txt")
        if not os.path.exists(method_file):
            with open(method_file, 'w') as f:
                f.write("=" * 70 + "\n")
                f.write("MTRgenefiltertext\n")
                f.write("=" * 70 + "\n\n")
                f.write("## text\n\n")
                f.write("1. datatext\n")
                f.write(" - RNA-seq readstexttomicrobetextgenetext\n")
                f.write(" - generatebedgraph(coverage)and.bam.txt(readsinformation)\n\n")
                f.write("2. normalizetext (RPM)\n")
                f.write(" - statisticstexttotextmicrobereadstext(from.bam.txt)\n")
                f.write(" - Targetsample: textsamplesmicrobereads\n")
                f.write(" - Normalsample: 89samplestextmicrobereads\n")
                f.write(" - RPM = (coverage / microbereads) × 1,000,000\n")
                f.write(" - text: text(flagstat), onlytextmicrobetextreads\n\n")
                f.write("3. geneannotation\n")
                f.write(" - textGFFfile, textGTF\n")
                f.write(" - extractgenetext, ID, text, textannotation\n")
                f.write(" - deduplicate: textgenetextfeaturetext\n")
                f.write(" - filter: < 50bpgene\n\n")
                f.write("4. textcalculate\n")
                f.write(" - textgenes: \n")
                f.write(" a) calculateTargetandNormalaveragecoverage(RPMafter normalization)\n")
                f.write(" b) calculatecoveragetext(withcoveragetext/genetext)\n")
                f.write(" c) calculatefold change = (Target_RPM + 0.1) / (Normal_RPM + 0.1)\n\n")
                f.write("5. filtertext\n")
                f.write(f" - Targetcoveragetext ≥ {config.MIN_GENE_COVERAGE_FRACTION}\n")
                f.write(f" - Targetaveragecoverage ≥ {config.MIN_TARGET_COVERAGE} RPM\n")
                f.write(f" - Fold change ≥ {config.MIN_FOLD_CHANGE}\n")
                f.write(f" - genetext ≥ {config.MIN_GENE_LENGTH} bp\n\n")
                f.write("## Output filestext\n\n")
                f.write("CSVfilecolumn: \n")
                f.write(" - sample: sampleID\n")
                f.write(" - taxon: microbetext\n")
                f.write(" - gene_*: genetextandannotationinformation\n")
                f.write(" - target_rpm: Targetnormalizecoverage\n")
                f.write(" - normal_rpm: Normalnormalizecoverage\n")
                f.write(" - fold_change: text\n")
                f.write(" - *_cov_fraction: coveragetext\n\n")
                f.write("FNAfile: \n")
                f.write(" - textgenetextsequence(fromtextgenetextextract)\n")
                f.write(" - FASTA headertext: text, genetext, text, fold change, annotationtext\n\n")
                f.write("filetext: \n")
                f.write(" - {sample}_{taxon}_MTR_genes_{GFF/GTF}.csv\n")
                f.write(" - {sample}_{taxon}_MTR_genes_{GFF/GTF}.fna\n")
                f.write(" - filenameintextannotationtext(GFForGTF)\n\n")
                print(f"\nOK resultssaved: {csv_file}")
                print(f" annotationtext: {file_type.upper()} file")
                # extracttextsavesequence
                gene_sequences = extract_gene_sequences(df, taxon)
                if gene_sequences:
                    if sample_id:
                        fna_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes_{file_type.upper()}.fna")
                    else:
                        fna_file = os.path.join(output_dir, f"{taxon}_MTR_genes_all_samples_{file_type.upper()}.fna")
                        with open(fna_file, 'w') as f:
                            for gene_key, sequence in gene_sequences.items():
                                parts = gene_key.split('|')
                                sample = parts[0]
                                coords = parts[1]
                                # fromdfintextgeneinformation
                                mask = (df['sample'] == sample) & (df['seqid'] + '_' + df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str) == coords)
                                if mask.any():
                                    gene_info = df[mask].iloc[0]
                                    header = f">{coords} {gene_info['gene_name']} | {gene_info['product']} | FC={gene_info['fold_change']} | {file_type.upper()}"
                                else:
                                    header = f">{coords} | {file_type.upper()}"
                                    f.write(header + '\n')
                                    # textrows60text
                                    for i in range(0, len(sequence), 60):
                                        f.write(sequence[i:i+60] + '\n')
                                        print(f"OK sequencesaved: {fna_file}")
                                        print(f" successextract: {len(gene_sequences)}/{len(df)} genessequence")
                                        # textstatisticstext
                                        print(f"\n{'='*70}")
                                        print("resultstext")
                                        print(f"{'='*70}")
                                        print(f"microbe: {taxon}")
                                        print(f"sample count: {df['sample'].nunique()}")
                                        print(f"textgenetext: {len(df)}")
                                        print(f"\nFold Changetext:")
                                        print(df['fold_change'].describe())
                                        print(f"\nTarget RPMtext:")
                                        print(df['target_rpm'].describe())
                                        print(f"\nNormal RPMtext:")
                                        print(df['normal_rpm'].describe())
                                        print(f"\ntextsamplestatistics:")
                                        print(df.groupby('sample').size().sort_values(ascending=False))

# ==================== text ====================

def main():
    """text"""
    print("=" * 70)
    print("MTRgenefilter - RPMnormalizetext(microbereads)")
    print("=" * 70)
    print("\ntext:")
    print(f" minimum gene length: {config.MIN_GENE_LENGTH} bp")
    print(f" textTargetcoverage: {config.MIN_TARGET_COVERAGE} (RPM)")
    print(f" textFold Change: {config.MIN_FOLD_CHANGE}")
    print(f" genecoveragetext: {config.MIN_GENE_COVERAGE_FRACTION}")
    if config.MAX_NORMAL_COVERAGE:
        print(f" textNormalcoverage: {config.MAX_NORMAL_COVERAGE} (RPM)")
        print("\nnormalizetext:")
        print(" - onlytexttotextmicrobereadstextrowsnormalize")
        print(" - Target: from .bam.txt statisticsmicrobemapped reads")
        print(" - Normal: from merged_Normal.bam.txt statistics")
        print(" - RPM = (coverage / microbereads) × 1,000,000")
        print(" - textflagstat(textgenetextstatistics)")
        # text: onlyprocessB7sample
        print("\n" + "="*70)
        print("text: onlyprocessB7sample")
        print("="*70)
        # readmicrobe list
        taxon_df = pd.read_csv(os.path.join(config.BASE_DIR, config.TAXON_LIST))
        taxons = taxon_df['Taxon'].tolist()
        print(f"\nshared {len(taxons)} microbes, text...")
        # textmicrobesB7sample
        test_taxon = taxons[0]
        start_time = time.time()
        df, file_type = process_taxon(test_taxon, sample_id='B7')
        if not df.empty:
            save_results(df, test_taxon, file_type, sample_id='B7')
            elapsed = time.time() - start_time
            print(f"\ntext: {elapsed:.2f} text")
            print("\n" + "="*70)
            print("textcompleted!textcheckresults, textreasonableaftertextprocesstextdata")
            print("="*70)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
MTRgeneanalyze - textstatisticstext
==========================================

text: 1. textsequencetext(seqidtextmatched)
2. textGFFinformationextract(textN/A)
3. textstatisticstext(text/text)
4. generatedetailedtextreport
"""

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import poisson, nbinom
import os
import gzip
import re
from collections import defaultdict, Counter
from Bio import SeqIO

# ==================== 1. sequencetext ====================

def diagnose_sequence_mismatch(csv_file: str, fasta_file: str, gff_file: str):
    """
    textfortextwithtextgenetexttosequence
    output:     - GFFinseqidlist
    - textgenetextinseqidlist
    - textmatchedseqid
    - textsequencegenetext
    """
    print("=" * 70)
    print("sequencetextreport")
    print("=" * 70)
    # 1. readCSVingeneinformation
    df = pd.read_csv(csv_file)
    print(f"\ntextgenetext: {len(df)}")
    # 2. checktextgenetext
    print(f"\n[1] checktextgenetext...")
    genome_seqids = set()
    if os.path.exists(fasta_file):
        for record in SeqIO.parse(fasta_file, 'fasta'):
            genome_seqids.add(record.id)
            print(f" textgenetext {len(genome_seqids)} textsequence")
            print(f" sequenceIDtext: {list(genome_seqids)[:5]}")
        else:
            print(f" Failed textgenetextexists: {fasta_file}")
            return
        # 3. checkGFFinseqid
        print(f"\n[2] checkGFFannotation...")
        gff_seqids = set()
        open_func = gzip.open if gff_file.endswith('.gz') else open
        with open_func(gff_file, 'rt') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                fields = line.strip().split('\t')
                if len(fields) >= 9:
                    gff_seqids.add(fields[0])
                    print(f" GFFannotationtext {len(gff_seqids)} textseqid")
                    print(f" seqidtext: {list(gff_seqids)[:5]}")
                    # 4. text
                    print(f"\n[3] seqidmatchedanalyze...")
                    csv_seqids = set(df['seqid'].unique())
                    print(f" CSVintextseqidtext: {len(csv_seqids)}")
                    matched_seqids = csv_seqids & genome_seqids
                    unmatched_seqids = csv_seqids - genome_seqids
                    print(f" OK matchedseqid: {len(matched_seqids)}")
                    print(f" Failed textmatchedseqid: {len(unmatched_seqids)}")
                    if unmatched_seqids:
                        print(f"\n textmatchedseqidlist:")
                        for seqid in sorted(unmatched_seqids):
                            count = len(df[df['seqid'] == seqid])
                            print(f" - {seqid}: {count} genes")
                            # 5. statisticstextsequencegene
                            print(f"\n[4] textsequencegenetext...")
                            missing_genes = df[~df['seqid'].isin(genome_seqids)]
                            print(f" textsequencegenetext: {len(missing_genes)} / {len(df)}")
                            if len(missing_genes) > 0:
                                print(f"\n first10textsequencegene:")
                                cols = ['seqid', 'gene_start', 'gene_end', 'gene_name', 'fold_change']
                                print(missing_genes[cols].head(10).to_string(index=False))
                                # 6. checktext
                                print(f"\n[5] seqidtextanalyze...")
                                print(f"\n textgenetext:")
                                for seqid in list(genome_seqids)[:3]:
                                    print(f" {seqid}")
                                    print(f"\n GFFannotationtext:")
                                    for seqid in list(csv_seqids)[:3]:
                                        print(f" {seqid}")
                                        # 7. textmatched
                                        print(f"\n[6] textmatched...")
                                        potential_matches = {}
                                        for unmatched in unmatched_seqids:
                                            # textaftertext
                                            base = unmatched.split('.')[0]
                                            candidates = [g for g in genome_seqids if g.startswith(base)]
                                            if candidates:
                                                potential_matches[unmatched] = candidates
                                                if potential_matches:
                                                    print(f" found {len(potential_matches)} textmatched:")
                                                    for orig, candidates in list(potential_matches.items())[:5]:
                                                        print(f" {orig} -> {candidates}")
                                                    else:
                                                        print(f" not foundtextmatched")
                                                        print("\n" + "=" * 70)

# ==================== 2. textGFFtext ====================

def parse_gff_attributes_enhanced(attr_string: str) -> dict:
    """
    textGFFtext, textextracttextinformation
    """
    attributes = {}
    # textGFFformat: key=value
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            # extracttext
            fields_to_extract = {
                'gene_id': ['ID', 'gene', 'locus_tag', 'db_xref'],
                'gene_name': ['Name', 'gene', 'gene_synonym'],
                'product': ['product', 'note', 'description', 'function'],
            }
            result = {}
            for field, possible_keys in fields_to_extract.items():
                for key in possible_keys:
                    if key in attributes and attributes[key] not in ['', 'N/A', 'NA']:
                        result[field] = attributes[key]
                        break
                    if field not in result:
                        result[field] = 'N/A'
                        # textinformation
                        result['all_attrs'] = attributes
                        return result

def diagnose_gff_content(gff_file: str, n_lines: int = 100):
    """
    textGFFfiletext, textactualtext
    """
    print("\n" + "=" * 70)
    print("GFFtext")
    print("=" * 70)
    open_func = gzip.open if gff_file.endswith('.gz') else open
    # statisticsfeaturetext
    feature_types = Counter()
    attribute_keys = Counter()
    sample_records = []
    with open_func(gff_file, 'rt') as f:
        for i, line in enumerate(f):
            if line.startswith('#') or not line.strip():
                continue
            fields = line.strip().split('\t')
            if len(fields) >= 9:
                feature_type = fields[2]
                feature_types[feature_type] += 1
                # text
                attrs = parse_gff_attributes_enhanced(fields[8])
                for key in attrs['all_attrs'].keys():
                    attribute_keys[key] += 1
                    # savesample
                    if i < n_lines and feature_type in ['gene', 'CDS']:
                        sample_records.append({
                            'seqid': fields[0],
                            'type': feature_type,
                            'start': fields[3],
                            'end': fields[4],
                            'attributes': fields[8][:100] + '...' if len(fields[8]) > 100 else fields[8]
                        })
                        print(f"\n[1] Featuretextstatistics:")
                        for ftype, count in feature_types.most_common(10):
                            print(f" {ftype}: {count}")
                            print(f"\n[2] textstatistics (top 15):")
                            for key, count in attribute_keys.most_common(15):
                                print(f" {key}: {count}")
                                print(f"\n[3] samplerecord (first5textgene/CDS):")
                                for rec in sample_records[:5]:
                                    print(f"\n seqid: {rec['seqid']}")
                                    print(f" type: {rec['type']}")
                                    print(f" coordinates: {rec['start']}-{rec['end']}")
                                    print(f" attributes: {rec['attributes']}")
                                    print("\n" + "=" * 70)

# ==================== 3. statisticstext ====================

def calculate_poisson_pvalue(target_count: float, normal_count: float,     target_reads: int, normal_reads: int) -> float:
"""
text: texttargetcoveragetextnormal
text: - textreadstextfromtext
- H0: targetcoverage = normalcoverage(after normalization)
- H1: targetcoverage > normalcoverage
text: target_count: targettextcoverage(RPM)
normal_count: normaltextcoverage(RPM)
target_reads: targettextmicrobereads
normal_reads: normaltextmicrobereads
"""
# textRPMtextcount (text)
target_raw = target_count * target_reads / 1e6
normal_raw = normal_count * normal_reads / 1e6
# text: textnormaltext
expected_rate = normal_raw / normal_reads if normal_reads > 0 else 0
expected_count = expected_rate * target_reads
# text: observed >= target_rawtext
if expected_count > 0:
    pvalue = 1 - poisson.cdf(target_raw - 1, expected_count)
else:
    pvalue = 1.0 if target_raw == 0 else 0.0
    return pvalue

def calculate_nbinom_pvalue(target_count: float, normal_count: float,
    target_reads: int, normal_reads: int,
    dispersion: float = 0.1) -> float:
"""
text: text
textRNA-seqdata, textfor: - textandtext
- text
text: dispersion: text(0.1text, textDESeq2)
"""
target_raw = target_count * target_reads / 1e6
normal_raw = normal_count * normal_reads / 1e6
# text
expected_rate = normal_raw / normal_reads if normal_reads > 0 else 0
mu = expected_rate * target_reads
if mu <= 0:
    return 1.0 if target_raw == 0 else 0.0
    # text
    # var = mu + dispersion * mu^2
    var = mu + dispersion * mu * mu
    # textfortext (n, p)
    if var > mu:
        p = mu / var
        n = mu * mu / (var - mu)
        pvalue = 1 - nbinom.cdf(target_raw - 1, n, p)
    else:
        # textfortext
        pvalue = 1 - poisson.cdf(target_raw - 1, mu)
        return pvalue

def add_statistical_tests(df: pd.DataFrame, target_reads: int,     normal_reads: int) -> pd.DataFrame:
"""
forresultstextstatisticstext
returntextcolumnDataFrame:
- pvalue_poisson: textPtext
- pvalue_nbinom: textPtext
- padj_poisson: BHtextafterPtext(text)
- padj_nbinom: BHtextafterPtext(text)
- significant: text(padj < 0.05)
"""
print("\n" + "=" * 70)
print("textstatisticstext")
print("=" * 70)
# calculatePtext
print("\n[1] calculatePtext...")
pvalues_poisson = []
pvalues_nbinom = []
for _, row in df.iterrows():
    # text
    p_poisson = calculate_poisson_pvalue(
        row['target_rpm'], row['normal_rpm'],
        target_reads, normal_reads
)
pvalues_poisson.append(p_poisson)
# text
p_nbinom = calculate_nbinom_pvalue(
    row['target_rpm'], row['normal_rpm'],
    target_reads, normal_reads
)
pvalues_nbinom.append(p_nbinom)
df['pvalue_poisson'] = pvalues_poisson
df['pvalue_nbinom'] = pvalues_nbinom
# Benjamini-Hochbergtext
print("[2] textrowstext(Benjamini-Hochberg)...")
from statsmodels.stats.multitest import multipletests
_, padj_poisson, _, _ = multipletests(pvalues_poisson, method='fdr_bh')
_, padj_nbinom, _, _ = multipletests(pvalues_nbinom, method='fdr_bh')
df['padj_poisson'] = padj_poisson
df['padj_nbinom'] = padj_nbinom
# text
df['significant_poisson'] = df['padj_poisson'] < 0.05
df['significant_nbinom'] = df['padj_nbinom'] < 0.05
# statistics
print(f"\n[3] textstatistics:")
print(f" textgenetext: {len(df)}")
print(f" text (padj<0.05): {df['significant_poisson'].sum()}")
print(f" text (padj<0.05): {df['significant_nbinom'].sum()}")
print(f" text: {(df['significant_poisson'] & df['significant_nbinom']).sum()}")
print(f"\n[4] Ptext:")
print("\n text:")
print(df['pvalue_poisson'].describe())
print("\n text:")
print(df['pvalue_nbinom'].describe())
return df

# ==================== 4. text ====================

def run_full_diagnostics(sample_id: str, taxon: str, base_dir: str):
    """
    textrowstext
    """
    print("\n" + "=" * 70)
    print(f"MTRgeneanalyze - textreport")
    print(f"sample: {sample_id}, microbe: {taxon}")
    print("=" * 70)
    # buildfilepath
    output_dir = os.path.join(base_dir, "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R/reports")
    csv_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes_GFF.csv")
    fna_file = os.path.join(output_dir, f"{sample_id}_{taxon}_MTR_genes_GFF.fna")
    reference_file = os.path.join(base_dir, f"data/reference/{taxon}.fna")
    gff_file = os.path.join(base_dir, f"data/genome_annotation/gff_mic/{taxon}_*.gff.gz")
    import glob
    gff_files = glob.glob(gff_file)
    if gff_files:
        gff_file = gff_files[0]
    else:
        print(f"error: not foundGFFfile")
        return
    # 1. sequencetext
    print("\n" + "=" * 70)
    print("text: sequencetext")
    print("=" * 70)
    diagnose_sequence_mismatch(csv_file, reference_file, gff_file)
    # 2. GFFtext
    print("\n" + "=" * 70)
    print("text: GFFannotationtextanalyze")
    print("=" * 70)
    diagnose_gff_content(gff_file)
    # 3. textstatisticstext
    print("\n" + "=" * 70)
    print("text: statisticstext")
    print("=" * 70)
    # readCSV
    df = pd.read_csv(csv_file)
    # textfromtextdatatextreadstext
    target_bam_txt = os.path.join(base_dir,         f"results_V3/special/CRC/03.coverage/R/{taxon}/{sample_id}_rna_output.pathseq_sorted.bam.txt")
    normal_bam_txt = os.path.join(base_dir,
        f"results_V3/cancers_V3.1/CRC/03.coverage/Normal/{taxon}/merged_Normal_sorted.bam.txt")
    # statisticsreadstext
    def count_reads(file_path):
        if not os.path.exists(file_path):
            return 0
        with open(file_path) as f:
            return sum(1 for line in f if not line.startswith('@'))
        target_reads = count_reads(target_bam_txt)
        normal_reads = count_reads(normal_bam_txt)
        print(f"\nReadsstatistics:")
        print(f" Target ({sample_id}): {target_reads:,} reads")
        print(f" Normal (merged): {normal_reads:,} reads")
        if target_reads > 0 and normal_reads > 0:
            df_with_stats = add_statistical_tests(df, target_reads, normal_reads)
            # savetextCSV
            output_file = os.path.join(output_dir,                 f"{sample_id}_{taxon}_MTR_genes_GFF_with_stats.csv")
            df_with_stats.to_csv(output_file, index=False)
            print(f"\nOK textresultssaved: {output_file}")
            # textgenetop10
            print(f"\n[5] text10genes (textPtext):")
            sig_cols = ['gene_name', 'product', 'fold_change', 'target_rpm',                 'pvalue_nbinom', 'padj_nbinom']
            top_sig = df_with_stats.nsmallest(10, 'pvalue_nbinom')[sig_cols]
            print(top_sig.to_string(index=False))
            print("\n" + "=" * 70)
            print("textcompleted!")
            print("=" * 70)

# ==================== text ====================

if __name__ == "__main__":
    # text
    BASE_DIR = "/path/to/project"
    SAMPLE_ID = "B7"
    TAXON = "Fusobacterium_animalis"
    # textrowstext
    run_full_diagnostics(SAMPLE_ID, TAXON, BASE_DIR)
    print("\n" + "=" * 70)
    print("text: ")
    print("=" * 70)
    print("""
    1. sequencetext:     - checkGFFandtextgenetext
    - textIDmappingfileortext
    - text, text(104/150text)
    2. GFFinformationextract:     - textN/AtextfortextGFFintextexists
    - textfromtext(textnote, function)extract
    - ortextdatatextannotation
    3. statisticstext:     - text(text)
    - padj < 0.05 textfortext
    - textfold changeandPtext
    4. text:     - textgenetextrowstextanalyze
    - checkgene cluster/text
    - textcoveragetext
    """)

In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - text
====================================

text: 1. OK textGFFtext - textfromCDSextractproduct
2. OK textstatisticstext(text)
3. OK textsequenceextractbug(deduplicate + textcheck)
4. OK textCSVsave(NaN → 'N/A')
5. OK detailedtextandtextinformation
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict, Counter
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from scipy import stats
from scipy.stats import nbinom, poisson
from statsmodels.stats.multitest import multipletests
import time

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50
    MIN_TARGET_COVERAGE: float = 0.5
    MIN_FOLD_CHANGE: float = 1.5
    MAX_NORMAL_COVERAGE: float = None
    MIN_GENE_COVERAGE_FRACTION: float = 0.3
    # statisticstext
    PVALUE_THRESHOLD: float = 0.05
    DISPERSION: float = 0.1 # text
    def __post_init__(self):
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'NORMAL_COVERAGE_DIR', 'OUTPUT_DIR']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== textGFFtext ====================

def parse_gff_attributes_enhanced(attr_string: str, feature_type: str) -> dict:
    """
    textGFFtext
    text:     1. text
    2. fromtextextract
    3. text
    """
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            result = {}
            # geneID
            for key in ['ID', 'gene', 'locus_tag', 'protein_id']:
                if key in attributes and attributes[key]:
                    result['gene_id'] = attributes[key]
                    break
                else:
                    result['gene_id'] = f"unknown_{feature_type}"
                    # genetext
                    for key in ['Name', 'gene', 'gene_synonym']:
                        if key in attributes and attributes[key]:
                            result['gene_name'] = attributes[key]
                            break
                        else:
                            result['gene_name'] = attributes.get('locus_tag', result['gene_id'])
                            # textannotation(text!)
                            product_value = None
                            product_sources = ['product', 'note', 'function', 'description']
                            for key in product_sources:
                                if key in attributes and attributes[key]:
                                    value = attributes[key]
                                    # skipnotext, textfortextaftertext
                                    if value not in ['hypothetical protein', 'unknown']:
                                        product_value = value
                                        break
                                    elif product_value is None:
                                        product_value = value
                                        # frominferencetext
                                        if not product_value and 'inference' in attributes:
                                            inference = attributes['inference']
                                            if 'similar to' in inference:
                                                product_value = f"similar to {inference.split(':')[-1]}"
                                                result['product'] = product_value if product_value else 'hypothetical protein'
                                                result['locus_tag'] = attributes.get('locus_tag', 'N/A')
                                                result['protein_id'] = attributes.get('protein_id', 'N/A')
                                                return result

def load_annotations_enhanced(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """
    textannotationload
    text:     1. textprocessgeneandCDS
    2. CDSproducttextgene
    3. textinformation
    """
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    # text: textallfeature
    all_features = defaultdict(lambda: {'gene': None, 'cds': []})
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            feature_type = fields[2]
            if feature_type not in ['gene', 'CDS']:
                continue
            attrs = parse_gff_attributes_enhanced(fields[8], feature_type)
            gene_info = {
                'seqid': fields[0],
                'type': feature_type,
                'start': int(fields[3]),
                'end': int(fields[4]),
                'strand': fields[6],
                'length': int(fields[4]) - int(fields[3]),
                **attrs
            }
            # textlocus_tagorgene_idtextforkey
            key = attrs.get('locus_tag', attrs.get('gene_id'))
            if feature_type == 'gene':
                all_features[key]['gene'] = gene_info
            else: # CDS
                all_features[key]['cds'].append(gene_info)
                # text: textinformation(CDSproducttext)
                for key, features in all_features.items():
                    gene_feature = features['gene']
                    cds_features = features['cds']
                    if gene_feature:
                        # textwithCDS, textCDSproduct
                        if cds_features:
                            best_cds = max(cds_features, key=lambda x: x['length'])
                            if best_cds['product'] != 'hypothetical protein':
                                gene_feature['product'] = best_cds['product']
                                if best_cds['protein_id'] != 'N/A':
                                    gene_feature['protein_id'] = best_cds['protein_id']
                                    annotations[gene_feature['seqid']].append(gene_feature)
                                elif cds_features:
                                    # onlywithCDS, text
                                    best_cds = max(cds_features, key=lambda x: x['length'])
                                    annotations[best_cds['seqid']].append(best_cds)
                                    return annotations

# ==================== statisticstext ====================

def calculate_pvalue_nbinom(target_rpm: float, normal_rpm: float,
    target_reads: int, normal_reads: int,
    gene_length: int, dispersion: float = 0.1) -> float:
"""
text
"""
# textforcount
target_count = (target_rpm * target_reads / 1e6)
normal_count = (normal_rpm * normal_reads / 1e6)
# text
size_factor = target_reads / normal_reads
expected_mean = normal_count * size_factor
if expected_mean <= 0:
    return 1.0 if target_count == 0 else 0.0
    # text
    expected_var = expected_mean + dispersion * expected_mean ** 2
    if expected_var > expected_mean:
        p = expected_mean / expected_var
        n = expected_mean ** 2 / (expected_var - expected_mean)
        pvalue = 1 - nbinom.cdf(target_count - 1, n, p)
    else:
        pvalue = 1 - poisson.cdf(target_count - 1, expected_mean)
        return pvalue

def add_statistical_tests(df: pd.DataFrame, target_reads: int,     normal_reads: int, dispersion: float = 0.1) -> pd.DataFrame:
"""textstatisticstext"""
print(f"\ncalculatestatisticstext(text, dispersion={dispersion})...")
pvalues = []
for _, row in df.iterrows():
    pval = calculate_pvalue_nbinom(
        row['target_rpm'], row['normal_rpm'],
        target_reads, normal_reads,
        row['gene_length'], dispersion
)
pvalues.append(pval)
df['pvalue'] = pvalues
# BHtext
_, padj, _, _ = multipletests(pvalues, method='fdr_bh')
df['padj'] = padj
df['significant'] = df['padj'] < config.PVALUE_THRESHOLD
n_sig = df['significant'].sum()
print(f" textgene (padj < {config.PVALUE_THRESHOLD}): {n_sig} / {len(df)}")
return df

# ==================== textUtility functions(text)====================

def get_microbial_mapped_reads(bam_txt_file: str) -> int:
    if not os.path.exists(bam_txt_file):
        return 0
    read_count = 0
    with open(bam_txt_file, 'r') as f:
        for line in f:
            if not line.startswith('@'):
                read_count += 1
                return read_count

def get_normal_total_microbial_reads(taxon: str) -> int:
    bam_txt = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,         'merged_Normal_sorted.bam.txt')
    total_reads = get_microbial_mapped_reads(bam_txt)
    if total_reads > 0:
        print(f" Normal: {total_reads:,} reads")
        return total_reads

def load_bedgraph(bedgraph_file: str, normalize_factor: float = 1.0):
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                regions[parts[0]].append((
                    int(parts[1]), int(parts[2]),                     float(parts[3]) * normalize_factor
))
return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
gene_length = gene_end - gene_start
coverage_array = np.zeros(gene_length, dtype=float)
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        covered_bases = np.sum(coverage_array > 0)
        return {
        'mean_coverage': np.mean(coverage_array),
        'covered_bases': int(covered_bases),
        'coverage_fraction': covered_bases / gene_length if gene_length > 0 else 0
    }

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    gene_dict = {}
    for gene in genes:
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict or gene['length'] > gene_dict[gene_id]['length']:
            gene_dict[gene_id] = gene
            return list(gene_dict.values())

# ==================== textsequenceextract ====================

def extract_gene_sequences_fixed(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """
    textsequenceextract
    text:     1. deduplicategene
    2. checktext
    3. detailedtext
    """
    print(f"\nextractgenesequence...")
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        print(f" warning: textgenetextexists")
        return {}
    # loadgenetext
    genome_seqs = {}
    genome_lengths = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        genome_lengths[record.id] = len(record.seq)
        print(f" loadtext {len(genome_seqs)} textsequence")
        # deduplicate
        df['gene_key'] = (df['sample'] + '|' + df['seqid'] + '_' +             df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str))
        df_unique = df.drop_duplicates('gene_key')
        if len(df_unique) < len(df):
            print(f" deduplicate: {len(df)} → {len(df_unique)} uniquegene")
            # extractsequence
            gene_sequences = {}
            failed = {'not_found': 0, 'out_of_range': 0, 'other': 0}
            for _, row in df_unique.iterrows():
                seqid = row['seqid']
                start = row['gene_start']
                end = row['gene_end']
                if seqid not in genome_seqs:
                    failed['not_found'] += 1
                    continue
                # checktext
                seq_len = genome_lengths[seqid]
                if start < 1 or end > seq_len:
                    failed['out_of_range'] += 1
                    continue
                try:
                    sequence = genome_seqs[seqid][start-1:end]
                    if len(sequence) > 0:
                        gene_sequences[row['gene_key']] = sequence
                    else:
                        failed['other'] += 1
                except Exception as e:
                    failed['other'] += 1
                    print(f" successextract: {len(gene_sequences)} / {len(df_unique)}")
                    if sum(failed.values()) > 0:
                        print(f" failedtext:")
                        for reason, count in failed.items():
                            if count > 0:
                                print(f" {reason}: {count}")
                                return gene_sequences

# ==================== textprocesstext ====================

def process_taxon(taxon: str, sample_id: str = None):
    print(f"\n{'='*70}")
    print(f"processmicrobe: {taxon}")
    print(f"{'='*70}")
    # 1. loadannotation(text)
    print("\n[1/6] loadgeneannotation(text)...")
    import glob
    gff_files = glob.glob(os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz"))
    gtf_files = glob.glob(os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz"))
    if gff_files:
        print(f" textGFF: {os.path.basename(gff_files[0])}")
        annotations = load_annotations_enhanced(gff_files[0], 'gff')
        file_type = 'gff'
    elif gtf_files:
        print(f" textGTF: {os.path.basename(gtf_files[0])}")
        annotations = load_annotations_enhanced(gtf_files[0], 'gtf')
        file_type = 'gtf'
    else:
        print(f" error: not foundannotationfile")
        return pd.DataFrame(), None, 0, 0
    all_genes = [gene for genes in annotations.values() for gene in genes]
    unique_genes = deduplicate_genes(all_genes)
    print(f" loadtext {len(unique_genes)} genes")
    # statisticsproducttext
    product_stats = Counter(g.get('product', 'N/A') for g in unique_genes)
    valid_products = sum(1 for p in product_stats if p not in ['N/A', 'hypothetical protein'])
    print(f" withannotationgene: {valid_products} / {len(unique_genes)}")
    # 2-5. processcoverageandtext(text)
    print("\n[2/6] calculateNormal reads...")
    normal_microbial_reads = get_normal_total_microbial_reads(taxon)
    if normal_microbial_reads == 0:
        return pd.DataFrame(), None, 0, 0
    normal_norm_factor = 1e6 / normal_microbial_reads
    print("\n[3/6] loadNormalcoverage...")
    normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,         'coverage_Normal.bedgraph')
    normal_coverage = load_bedgraph(normal_bedgraph, normal_norm_factor)
    print("\n[4/6] findTargetsample...")
    target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
    if not os.path.exists(target_dir):
        return pd.DataFrame(), None, 0, 0
    bedgraph_files = [f for f in os.listdir(target_dir)         if f.endswith('_rna_output.pathseq.bedgraph')]
    if sample_id:
        bedgraph_files = [f for f in bedgraph_files if f.startswith(f"{sample_id}_")]
        print(f" found {len(bedgraph_files)} samples")
        print("\n[5/6] calculatetext...")
        all_results = []
        target_microbial_reads = 0
        for bedgraph_file in bedgraph_files:
            sample_name = bedgraph_file.replace('_rna_output.pathseq.bedgraph', '')
            target_bam_txt = os.path.join(target_dir,                 f"{sample_name}_rna_output.pathseq_sorted.bam.txt")
            target_microbial_reads = get_microbial_mapped_reads(target_bam_txt)
            if target_microbial_reads == 0:
                continue
            print(f" {sample_name}: {target_microbial_reads:,} reads")
            target_norm_factor = 1e6 / target_microbial_reads
            target_bedgraph = os.path.join(target_dir, bedgraph_file)
            target_coverage = load_bedgraph(target_bedgraph, target_norm_factor)
            for gene in unique_genes:
                target_stats = calculate_gene_coverage(
                    gene['start'], gene['end'],
                    target_coverage.get(gene['seqid'], [])
)
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(gene['seqid'], [])
)
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue
    all_results.append({
        'sample': sample_name,
        'taxon': taxon,
        'seqid': gene['seqid'],
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        'locus_tag': gene.get('locus_tag', 'N/A'),
        'protein_id': gene.get('protein_id', 'N/A'),
        'target_rpm': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        'normal_rpm': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        'fold_change': round(fold_change, 2),
        'coverage_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    })
    print(f"\n[6/6] found {len(all_results)} textgene")
    if not all_results:
        return pd.DataFrame(), None, 0, 0
    df = pd.DataFrame(all_results)
    df = df.sort_values(['sample', 'fold_change'], ascending=[True, False])
    return df, file_type, target_microbial_reads, normal_microbial_reads

# ==================== saveresults(textCSVsave)====================

def save_results(df: pd.DataFrame, taxon: str, file_type: str,     target_reads: int, normal_reads: int, sample_id: str = None):
if df.empty:
    return
    output_dir = os.path.join(config.OUTPUT_DIR, 'reports')
    os.makedirs(output_dir, exist_ok=True)
    # textstatisticstext
    df = add_statistical_tests(df, target_reads, normal_reads, config.DISPERSION)
    # saveCSV(text: na_rep='N/A')
    csv_file = os.path.join(output_dir,         f"{sample_id}_{taxon}_MTR_genes_{file_type.upper()}_enhanced.csv" if sample_id     else f"{taxon}_MTR_genes_all_samples_{file_type.upper()}_enhanced.csv")
df.to_csv(csv_file, index=False, na_rep='N/A') # text!
print(f"\nOK resultssaved: {csv_file}")
# extractsequence(text)
gene_sequences = extract_gene_sequences_fixed(df, taxon)
if gene_sequences:
    fna_file = csv_file.replace('.csv', '.fna')
    with open(fna_file, 'w') as f:
        for gene_key, sequence in gene_sequences.items():
            parts = gene_key.split('|')
            coords = parts[1]
            # findgeneinformation
            match = df[df['gene_key'] == gene_key]
            if not match.empty:
                row = match.iloc[0]
                header = (f">{coords} {row['gene_name']} | {row['product']} | "
                    f"FC={row['fold_change']} | padj={row['padj']:.2e} | {file_type.upper()}")
            else:
                header = f">{coords} | {file_type.upper()}"
                f.write(header + '\n')
                for i in range(0, len(sequence), 60):
                    f.write(sequence[i:i+60] + '\n')
                    print(f"OK sequencesaved: {fna_file}")
                    # statisticstext
                    print(f"\n{'='*70}")
                    print("resultstext")
                    print(f"{'='*70}")
                    print(f"textgenetext: {len(df)}")
                    print(f"textgenetext (padj<0.05): {df['significant'].sum()}")
                    print(f"\ntext10genes:")
                    top_cols = ['gene_name', 'product', 'fold_change', 'target_rpm', 'pvalue', 'padj']
                    print(df.nsmallest(10, 'pvalue')[top_cols].to_string(index=False))

# ==================== text ====================

def main():
    print("=" * 70)
    print("MTRgenefilter - text")
    print("=" * 70)
    print("\ntext:")
    print(" OK textGFFtext(CDS producttext)")
    print(" OK textstatisticstext(text)")
    print(" OK textsequenceextractbug")
    print(" OK textCSV NaNtext")
    taxon_df = pd.read_csv(os.path.join(config.BASE_DIR, config.TAXON_LIST))
    test_taxon = taxon_df['Taxon'].tolist()[0]
    start_time = time.time()
    df, file_type, target_reads, normal_reads = process_taxon(test_taxon, sample_id='B7')
    if not df.empty:
        save_results(df, test_taxon, file_type, target_reads, normal_reads, sample_id='B7')
        print(f"\ntext: {time.time() - start_time:.2f} text")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
MTRgenefilter - textprocesstext
====================================

text: 1. textprocessallmicrobe
2. textmicrobesprocessallTargetsample
3. outputtext: R/{Taxon}/{SampleID}_{Taxon}_MTR_genes_*.csv/fna
4. textanderror while processing
5. text(skipprocessedsample)
"""

import pandas as pd
import numpy as np
import os
import gzip
import re
from collections import defaultdict, Counter
from Bio import SeqIO
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from scipy import stats
from scipy.stats import nbinom, poisson
from statsmodels.stats.multitest import multipletests
import time
from datetime import datetime
import traceback

# ==================== text ====================
@dataclass
class Config:
    """Configuration parameters"""
    BASE_DIR: str = "/path/to/project"
    TAXON_LIST: str = "data/CSVs_20251114/CRC_R_NR_paired.csv"
    REFERENCE_DIR: str = "data/reference"
    GFF_DIR: str = "data/genome_annotation/gff_mic"
    GTF_DIR: str = "data/genome_annotation/gtf_mic"
    TARGET_COVERAGE_DIR: str = "results_V3/special/CRC/03.coverage/R"
    NORMAL_COVERAGE_DIR: str = "results_V3/cancers_V3.1/CRC/03.coverage/Normal"
    OUTPUT_DIR: str = "results_V3/special/CRC/05.MTR/05.2.region_gene_based/R"
    # filtertext
    MIN_GENE_LENGTH: int = 50
    MIN_TARGET_COVERAGE: float = 0.5
    MIN_FOLD_CHANGE: float = 1.5
    MAX_NORMAL_COVERAGE: float = None
    MIN_GENE_COVERAGE_FRACTION: float = 0.3
    # statisticstext
    PVALUE_THRESHOLD: float = 0.05
    DISPERSION: float = 0.1
    # textprocesstext
    SKIP_EXISTING: bool = True # skipprocessedsample
    LOG_FILE: str = "MTR_batch_processing.log"
    def __post_init__(self):
        for field in ['TAXON_LIST', 'REFERENCE_DIR', 'GFF_DIR', 'GTF_DIR',
            'TARGET_COVERAGE_DIR', 'NORMAL_COVERAGE_DIR', 'OUTPUT_DIR', 'LOG_FILE']:
        value = getattr(self, field)
        if not value.startswith('/'):
            setattr(self, field, os.path.join(self.BASE_DIR, value))

config = Config()

# ==================== text ====================

class Logger:
    """text"""
    def __init__(self, log_file: str):
        self.log_file = log_file
        self.start_time = time.time()
        def log(self, message: str, level: str = "INFO"):
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            elapsed = time.time() - self.start_time
            log_msg = f"[{timestamp}] [{level}] [{elapsed:.1f}s] {message}"
            print(log_msg)
            with open(self.log_file, 'a') as f:
                f.write(log_msg + '\n')
                def error(self, message: str):
                    self.log(message, "ERROR")
                    def warning(self, message: str):
                        self.log(message, "WARN")
                        def info(self, message: str):
                            self.log(message, "INFO")

logger = Logger(config.LOG_FILE)

# ==================== textGFFtext ====================

def parse_gff_attributes_enhanced(attr_string: str, feature_type: str) -> dict:
    """textGFFtext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            result = {}
            # geneID
            for key in ['ID', 'gene', 'locus_tag', 'protein_id']:
                if key in attributes and attributes[key]:
                    result['gene_id'] = attributes[key]
                    break
                else:
                    result['gene_id'] = f"unknown_{feature_type}"
                    # genetext
                    for key in ['Name', 'gene', 'gene_synonym']:
                        if key in attributes and attributes[key]:
                            result['gene_name'] = attributes[key]
                            break
                        else:
                            result['gene_name'] = attributes.get('locus_tag', result['gene_id'])
                            # textannotation
                            product_value = None
                            for key in ['product', 'note', 'function', 'description']:
                                if key in attributes and attributes[key]:
                                    value = attributes[key]
                                    if value not in ['hypothetical protein', 'unknown']:
                                        product_value = value
                                        break
                                    elif product_value is None:
                                        product_value = value
                                        result['product'] = product_value if product_value else 'hypothetical protein'
                                        result['locus_tag'] = attributes.get('locus_tag', 'N/A')
                                        result['protein_id'] = attributes.get('protein_id', 'N/A')
                                        return result

def load_annotations_enhanced(annotation_file: str, file_type: str) -> Dict[str, List[Dict]]:
    """textannotationload"""
    annotations = defaultdict(list)
    if not os.path.exists(annotation_file):
        return annotations
    open_func = gzip.open if annotation_file.endswith('.gz') else open
    all_features = defaultdict(lambda: {'gene': None, 'cds': []})
    with open_func(annotation_file, 'rt') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            feature_type = fields[2]
            if feature_type not in ['gene', 'CDS']:
                continue
            attrs = parse_gff_attributes_enhanced(fields[8], feature_type)
            gene_info = {
                'seqid': fields[0],
                'type': feature_type,
                'start': int(fields[3]),
                'end': int(fields[4]),
                'strand': fields[6],
                'length': int(fields[4]) - int(fields[3]),
                **attrs
            }
            key = attrs.get('locus_tag', attrs.get('gene_id'))
            if feature_type == 'gene':
                all_features[key]['gene'] = gene_info
            else:
                all_features[key]['cds'].append(gene_info)
                # textinformation
                for key, features in all_features.items():
                    gene_feature = features['gene']
                    cds_features = features['cds']
                    if gene_feature:
                        if cds_features:
                            best_cds = max(cds_features, key=lambda x: x['length'])
                            if best_cds['product'] != 'hypothetical protein':
                                gene_feature['product'] = best_cds['product']
                                if best_cds['protein_id'] != 'N/A':
                                    gene_feature['protein_id'] = best_cds['protein_id']
                                    annotations[gene_feature['seqid']].append(gene_feature)
                                elif cds_features:
                                    best_cds = max(cds_features, key=lambda x: x['length'])
                                    annotations[best_cds['seqid']].append(best_cds)
                                    return annotations

# ==================== statisticstext ====================

def calculate_pvalue_nbinom(target_rpm: float, normal_rpm: float,
    target_reads: int, normal_reads: int,
    gene_length: int, dispersion: float = 0.1) -> float:
"""text"""
target_count = (target_rpm * target_reads / 1e6)
normal_count = (normal_rpm * normal_reads / 1e6)
size_factor = target_reads / normal_reads
expected_mean = normal_count * size_factor
if expected_mean <= 0:
    return 1.0 if target_count == 0 else 0.0
    expected_var = expected_mean + dispersion * expected_mean ** 2
    if expected_var > expected_mean:
        p = expected_mean / expected_var
        n = expected_mean ** 2 / (expected_var - expected_mean)
        pvalue = 1 - nbinom.cdf(target_count - 1, n, p)
    else:
        pvalue = 1 - poisson.cdf(target_count - 1, expected_mean)
        return pvalue

def add_statistical_tests(df: pd.DataFrame, target_reads: int,     normal_reads: int, dispersion: float = 0.1) -> pd.DataFrame:
"""textstatisticstext"""
pvalues = []
for _, row in df.iterrows():
    pval = calculate_pvalue_nbinom(
        row['target_rpm'], row['normal_rpm'],
        target_reads, normal_reads,
        row['gene_length'], dispersion
)
pvalues.append(pval)
df['pvalue'] = pvalues
if len(pvalues) > 0:
    _, padj, _, _ = multipletests(pvalues, method='fdr_bh')
    df['padj'] = padj
else:
    df['padj'] = []
    df['significant'] = df['padj'] < config.PVALUE_THRESHOLD
    return df

# ==================== Utility functions ====================

def get_microbial_mapped_reads(bam_txt_file: str) -> int:
    if not os.path.exists(bam_txt_file):
        return 0
    read_count = 0
    with open(bam_txt_file, 'r') as f:
        for line in f:
            if not line.startswith('@'):
                read_count += 1
                return read_count

def load_bedgraph(bedgraph_file: str, normalize_factor: float = 1.0):
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                regions[parts[0]].append((
                    int(parts[1]), int(parts[2]),                     float(parts[3]) * normalize_factor
))
return regions

def calculate_gene_coverage(gene_start: int, gene_end: int,     bedgraph_regions: List[Tuple[int, int, float]]) -> Dict:
gene_length = gene_end - gene_start
coverage_array = np.zeros(gene_length, dtype=float)
for bg_start, bg_end, bg_coverage in bedgraph_regions:
    overlap_start = max(gene_start, bg_start)
    overlap_end = min(gene_end, bg_end)
    if overlap_start < overlap_end:
        rel_start = overlap_start - gene_start
        rel_end = overlap_end - gene_start
        coverage_array[rel_start:rel_end] = bg_coverage
        covered_bases = np.sum(coverage_array > 0)
        return {
        'mean_coverage': np.mean(coverage_array),
        'covered_bases': int(covered_bases),
        'coverage_fraction': covered_bases / gene_length if gene_length > 0 else 0
    }

def deduplicate_genes(genes: List[Dict]) -> List[Dict]:
    gene_dict = {}
    for gene in genes:
        if gene['length'] < config.MIN_GENE_LENGTH:
            continue
        gene_id = gene.get('gene_id', f"{gene['seqid']}_{gene['start']}_{gene['end']}")
        if gene_id not in gene_dict or gene['length'] > gene_dict[gene_id]['length']:
            gene_dict[gene_id] = gene
            return list(gene_dict.values())

def extract_gene_sequences_fixed(df: pd.DataFrame, taxon: str) -> Dict[str, str]:
    """textsequenceextract"""
    ref_file = os.path.join(config.REFERENCE_DIR, f"{taxon}.fna")
    if not os.path.exists(ref_file):
        return {}
    genome_seqs = {}
    genome_lengths = {}
    for record in SeqIO.parse(ref_file, 'fasta'):
        genome_seqs[record.id] = str(record.seq)
        genome_lengths[record.id] = len(record.seq)
        # deduplicate
        df['gene_key'] = (df['sample'] + '|' + df['seqid'] + '_' +             df['gene_start'].astype(str) + '_' + df['gene_end'].astype(str))
        df_unique = df.drop_duplicates('gene_key')
        gene_sequences = {}
        for _, row in df_unique.iterrows():
            seqid = row['seqid']
            start = row['gene_start']
            end = row['gene_end']
            if seqid not in genome_seqs:
                continue
            seq_len = genome_lengths[seqid]
            if start < 1 or end > seq_len:
                continue
            try:
                sequence = genome_seqs[seqid][start-1:end]
                if len(sequence) > 0:
                    gene_sequences[row['gene_key']] = sequence
            except:
                continue
            return gene_sequences

# ==================== textprocesstext ====================

def process_single_sample(sample_name: str, taxon: str,     unique_genes: List[Dict],
    normal_coverage: Dict,
    normal_microbial_reads: int,
    target_dir: str) -> Tuple[pd.DataFrame, int]:
"""
processtextsamples
return: (DataFrame, target_microbial_reads)
"""
# textTarget readstext
target_bam_txt = os.path.join(target_dir,     f"{sample_name}_rna_output.pathseq_sorted.bam.txt")
target_microbial_reads = get_microbial_mapped_reads(target_bam_txt)
if target_microbial_reads == 0:
    logger.warning(f" {sample_name}: notextreadstext, skip")
    return pd.DataFrame(), 0
    # loadTargetcoverage
    target_norm_factor = 1e6 / target_microbial_reads
    target_bedgraph = os.path.join(target_dir,         f"{sample_name}_rna_output.pathseq.bedgraph")
    target_coverage = load_bedgraph(target_bedgraph, target_norm_factor)
    # calculatetextgenes
    sample_results = []
    for gene in unique_genes:
        target_stats = calculate_gene_coverage(
            gene['start'], gene['end'],
            target_coverage.get(gene['seqid'], [])
)
normal_stats = calculate_gene_coverage(
    gene['start'], gene['end'],
    normal_coverage.get(gene['seqid'], [])
)
# filter
if target_stats['coverage_fraction'] < config.MIN_GENE_COVERAGE_FRACTION:
    continue
    if target_stats['mean_coverage'] < config.MIN_TARGET_COVERAGE:
        continue
    fold_change = (target_stats['mean_coverage'] + 0.1) / (normal_stats['mean_coverage'] + 0.1)
    if fold_change < config.MIN_FOLD_CHANGE:
        continue
    sample_results.append({
        'sample': sample_name,
        'taxon': taxon,
        'seqid': gene['seqid'],
        'gene_start': gene['start'],
        'gene_end': gene['end'],
        'gene_length': gene['length'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        'locus_tag': gene.get('locus_tag', 'N/A'),
        'protein_id': gene.get('protein_id', 'N/A'),
        'target_rpm': round(target_stats['mean_coverage'], 2),
        'target_covered_bases': target_stats['covered_bases'],
        'target_cov_fraction': round(target_stats['coverage_fraction'], 3),
        'normal_rpm': round(normal_stats['mean_coverage'], 2),
        'normal_covered_bases': normal_stats['covered_bases'],
        'normal_cov_fraction': round(normal_stats['coverage_fraction'], 3),
        'fold_change': round(fold_change, 2),
        'coverage_diff': round(target_stats['mean_coverage'] - normal_stats['mean_coverage'], 2),
    })
    if sample_results:
        df = pd.DataFrame(sample_results)
        df = df.sort_values('fold_change', ascending=False)
        return df, target_microbial_reads
    return pd.DataFrame(), target_microbial_reads

def process_taxon(taxon: str, sample_filter: List[str] = None) -> dict:
    """
    processtextmicrobesallsample
    text:     taxon: microbetext
    sample_filter: onlyprocesstextsample(None=text)
    return:     {'success': [], 'failed': [], 'skipped': []}
    """
    logger.info(f"\n{'='*70}")
    logger.info(f"processmicrobe: {taxon}")
    logger.info(f"{'='*70}")
    result_summary = {'success': [], 'failed': [], 'skipped': []}
    try:
        # 1. loadannotation
        import glob
        gff_files = glob.glob(os.path.join(config.GFF_DIR, f"{taxon}_*.gff.gz"))
        gtf_files = glob.glob(os.path.join(config.GTF_DIR, f"{taxon}_*.gtf.gz"))
        if gff_files:
            logger.info(f" textGFF: {os.path.basename(gff_files[0])}")
            annotations = load_annotations_enhanced(gff_files[0], 'gff')
            file_type = 'gff'
        elif gtf_files:
            logger.info(f" textGTF: {os.path.basename(gtf_files[0])}")
            annotations = load_annotations_enhanced(gtf_files[0], 'gtf')
            file_type = 'gtf'
        else:
            logger.error(f" not foundannotationfile")
            return result_summary
        all_genes = [gene for genes in annotations.values() for gene in genes]
        unique_genes = deduplicate_genes(all_genes)
        logger.info(f" loadtext {len(unique_genes)} genes")
        # 2. loadNormaldata
        normal_bam_txt = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'merged_Normal_sorted.bam.txt')
        normal_microbial_reads = get_microbial_mapped_reads(normal_bam_txt)
        if normal_microbial_reads == 0:
            logger.error(f" notextNormal readstext")
            return result_summary
        logger.info(f" Normal reads: {normal_microbial_reads:,}")
        normal_norm_factor = 1e6 / normal_microbial_reads
        normal_bedgraph = os.path.join(config.NORMAL_COVERAGE_DIR, taxon,             'coverage_Normal.bedgraph')
        normal_coverage = load_bedgraph(normal_bedgraph, normal_norm_factor)
        # 3. findTargetsample
        target_dir = os.path.join(config.TARGET_COVERAGE_DIR, taxon)
        if not os.path.exists(target_dir):
            logger.error(f" Targetdirectorytextexists: {target_dir}")
            return result_summary
        bedgraph_files = [f for f in os.listdir(target_dir)             if f.endswith('_rna_output.pathseq.bedgraph')]
        # extractsampleID
        samples = [f.replace('_rna_output.pathseq.bedgraph', '') for f in bedgraph_files]
        # filtersample
        if sample_filter:
            samples = [s for s in samples if s in sample_filter]
            logger.info(f" found {len(samples)} samples")
            # 4. Create output directory
            output_dir = os.path.join(config.OUTPUT_DIR, taxon)
            os.makedirs(output_dir, exist_ok=True)
            # 5. processtextsamples
            for idx, sample_name in enumerate(samples, 1):
                logger.info(f"\n [{idx}/{len(samples)}] Process sample: {sample_name}")
                # checktextexists
                csv_file = os.path.join(output_dir,                     f"{sample_name}_{taxon}_MTR_genes_{file_type.upper()}.csv")
                if config.SKIP_EXISTING and os.path.exists(csv_file):
                    logger.info(f" textexists, skip")
                    result_summary['skipped'].append(sample_name)
                    continue
                try:
                    # Process sample
                    df, target_reads = process_single_sample(
                        sample_name, taxon, unique_genes,
                        normal_coverage, normal_microbial_reads,
                        target_dir
)
if df.empty:
    logger.warning(f" not foundtextgene")
    result_summary['failed'].append(sample_name)
    continue
    logger.info(f" Target reads: {target_reads:,}")
    logger.info(f" found {len(df)} textgene")
    # textstatisticstext
    df = add_statistical_tests(df, target_reads, normal_microbial_reads,         config.DISPERSION)
    n_sig = df['significant'].sum()
    logger.info(f" textgene: {n_sig} (padj<0.05)")
    # saveCSV
    df.to_csv(csv_file, index=False, na_rep='N/A')
    logger.info(f" OK CSVsaved: {os.path.basename(csv_file)}")
    # extractsequence
    gene_sequences = extract_gene_sequences_fixed(df, taxon)
    if gene_sequences:
        fna_file = csv_file.replace('.csv', '.fna')
        with open(fna_file, 'w') as f:
            for gene_key, sequence in gene_sequences.items():
                parts = gene_key.split('|')
                coords = parts[1]
                match = df[df['gene_key'] == gene_key]
                if not match.empty:
                    row = match.iloc[0]
                    header = (f">{coords} {row['gene_name']} | {row['product']} | "
                        f"FC={row['fold_change']} | padj={row['padj']:.2e} | {file_type.upper()}")
                else:
                    header = f">{coords} | {file_type.upper()}"
                    f.write(header + '\n')
                    for i in range(0, len(sequence), 60):
                        f.write(sequence[i:i+60] + '\n')
                        logger.info(f" OK FNAsaved: {len(gene_sequences)} sequence")
                        result_summary['success'].append(sample_name)
                        except Exception as e:
                            logger.error(f" processfailed: {str(e)}")
                            logger.error(traceback.format_exc())
                            result_summary['failed'].append(sample_name)
                            # 6. textstatistics
                            logger.info(f"\n {'='*70}")
                            logger.info(f" {taxon} Processing completed:")
                            logger.info(f" success: {len(result_summary['success'])}")
                            logger.info(f" failed: {len(result_summary['failed'])}")
                            logger.info(f" skip: {len(result_summary['skipped'])}")
                        except Exception as e:
                            logger.error(f" microbeprocessfailed: {str(e)}")
                            logger.error(traceback.format_exc())
                            return result_summary

# ==================== textprocesstext ====================

def batch_process_all(taxon_filter: List[str] = None,     sample_filter: List[str] = None):
"""
textprocessallmicrobeandsample
text: taxon_filter: onlyprocesstextmicrobe(None=text)
sample_filter: onlyprocesstextsample(None=text)
"""
logger.info("=" * 70)
logger.info("MTRgenefilter - textprocess")
logger.info("=" * 70)
logger.info(f"\ntext:")
logger.info(f" minimum gene length: {config.MIN_GENE_LENGTH} bp")
logger.info(f" textTargetcoverage: {config.MIN_TARGET_COVERAGE} RPM")
logger.info(f" textFold Change: {config.MIN_FOLD_CHANGE}")
logger.info(f" genecoveragetext: {config.MIN_GENE_COVERAGE_FRACTION}")
logger.info(f" Ptext: {config.PVALUE_THRESHOLD}")
logger.info(f" skiptextexists: {config.SKIP_EXISTING}")
# readmicrobe list
taxon_df = pd.read_csv(config.TAXON_LIST)
taxons = taxon_df['Taxon'].tolist()
# filtermicrobe
if taxon_filter:
    taxons = [t for t in taxons if t in taxon_filter]
    logger.info(f"\nfilterafterprocess {len(taxons)} microbes")
else:
    logger.info(f"\ntext {len(taxons)} microbes")
    # textstatistics
    global_summary = {
        'taxons_success': [],
        'taxons_failed': [],
        'total_samples_success': 0,
        'total_samples_failed': 0,
        'total_samples_skipped': 0
    }
    start_time = time.time()
    # processtextmicrobes
    for idx, taxon in enumerate(taxons, 1):
        logger.info(f"\n\n{'#'*70}")
        logger.info(f"# [{idx}/{len(taxons)}] microbe: {taxon}")
        logger.info(f"{'#'*70}")
        try:
            result = process_taxon(taxon, sample_filter)
            if result['success'] or result['skipped']:
                global_summary['taxons_success'].append(taxon)
            else:
                global_summary['taxons_failed'].append(taxon)
                global_summary['total_samples_success'] += len(result['success'])
                global_summary['total_samples_failed'] += len(result['failed'])
                global_summary['total_samples_skipped'] += len(result['skipped'])
        except Exception as e:
            logger.error(f"microbe {taxon} processfailed: {str(e)}")
            global_summary['taxons_failed'].append(taxon)
            # text
            elapsed = time.time() - start_time
            logger.info(f"\n\n{'='*70}")
            logger.info("textProcessing completed!")
            logger.info(f"{'='*70}")
            logger.info(f"\ntext: {elapsed/60:.1f} text")
            logger.info(f"\nmicrobestatistics:")
            logger.info(f" success: {len(global_summary['taxons_success'])} / {len(taxons)}")
            logger.info(f" failed: {len(global_summary['taxons_failed'])}")
            logger.info(f"\nsamplestatistics:")
            logger.info(f" processed successfully: {global_summary['total_samples_success']}")
            logger.info(f" processfailed: {global_summary['total_samples_failed']}")
            logger.info(f" skiptextexists: {global_summary['total_samples_skipped']}")
            if global_summary['taxons_failed']:
                logger.info(f"\nfailedmicrobe:")
                for t in global_summary['taxons_failed']:
                    logger.info(f" - {t}")
                    logger.info(f"\noutput directory: {config.OUTPUT_DIR}")
                    logger.info(f"log files: {config.LOG_FILE}")

# # ==================== text ====================

# def main():
# """
# text - textrowstext
# """
# import sys
# if len(sys.argv) > 1:
# mode = sys.argv[1]
# if mode == 'test':
# # text: onlyprocesstextmicrobestextsamples
# logger.info("text: processtextmicrobestextsamples")
# taxon_df = pd.read_csv(config.TAXON_LIST)
# test_taxon = taxon_df['Taxon'].tolist()[0]
# # textsamples
# target_dir = os.path.join(config.TARGET_COVERAGE_DIR, test_taxon)
# if os.path.exists(target_dir):
# samples = [f.replace('_rna_output.pathseq.bedgraph', '') # for f in os.listdir(target_dir) # if f.endswith('_rna_output.pathseq.bedgraph')]
# if samples:
# batch_process_all(taxon_filter=[test_taxon], # sample_filter=[samples[0]])
# elif mode == 'single':
# # textmicrobestext
# if len(sys.argv) > 2:
# taxon = sys.argv[2]
# logger.info(f"textmicrobestext: {taxon}")
# batch_process_all(taxon_filter=[taxon])
# else:
# logger.error("textmicrobetext")
# elif mode == 'all':
# # textprocess
# logger.info("textprocesstext")
# batch_process_all()
# else:
# print("text:")
# print(" python script.py test # text")
# print(" python script.py single <taxon> # processtextmicrobes")
# print(" python script.py all # processallmicrobe")
# else:
# # text: textprocess
# logger.info("text: processallmicrobe")
# batch_process_all()

# if __name__ == "__main__":
# main()

In [ ]:
# textprocess
logger.info("textprocesstext")
batch_process_all()